# Group 12 — ACHP Medicare Advantage Market Analysis
## Data Ingestion & Master File Construction

Our analysis draws on four publicly available CMS data sources, linked together
via the ACHP member contract crosswalk provided by our sponsor.
The cells below build the two master files (`2025_master.csv`, `2026_master.csv`)
that the rest of the pipeline depends on.

**Input files required in `/content/`:**

| File | Description |
|------|-------------|
| `1. ACHP_Members_Crosswalk_2026_JHU (...).xlsx` | ACHP member contract IDs (2026) |
| `1. ACHP_Members_Crosswalk_2025_JHU (...).xlsx` | ACHP member contract IDs (2025) |
| `2. CY2026_Landscape_202603.csv` | CMS MA Landscape — plan attributes & cost sharing (2026) |
| `5. CY2025_Landscape_202506.1.csv` | CMS MA Landscape — plan attributes & cost sharing (2025) |
| `CPSC_Monthly/cpsc_2025_01.zip … cpsc_2025_12.zip` | Monthly enrollment ZIPs (12 files) |
| `4. 2026_HPMS_Display_Measures_12_04_2025.csv` | CMS display quality measures |
| `4. 2026_Report_Card_Master_Table_2025_10_08.xlsx` | CMS star ratings & performance data |

> **Skip to Cell 2 if you already have `2025_master.csv` and `2026_master.csv` in `/content/`.**


In [1]:
!pip install matplotlib seaborn statsmodels openpyxl scikit-learn -q

import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.stats import chi2_contingency
from openpyxl.drawing.image import Image as XLImage
warnings.filterwarnings('ignore')

OUT = '/content/'
print('✓ Libraries ready')

✓ Libraries ready


In [ ]:
# We configure all file paths up front so the rest of the build cells
# can be run in sequence without any manual edits.

from pathlib import Path
import zipfile, warnings
warnings.filterwarnings('ignore')

CONTENT = Path('/content')

ACHP_XWALK_2026    = CONTENT / '1. ACHP_Members_Crosswalk_2026_JHU (List of ACHP Member Plans).xlsx'
ACHP_XWALK_2025    = CONTENT / '1. ACHP_Members_Crosswalk_2025_JHU (List of ACHP Member Plans).xlsx'
LANDSCAPE_2026     = CONTENT / '2. CY2026_Landscape_202603.csv'
LANDSCAPE_2025     = CONTENT / '5. CY2025_Landscape_202506.1.csv'
CPSC_DIR           = CONTENT / 'CPSC_Monthly'
DISPLAY_MEASURES   = CONTENT / '4. 2026_HPMS_Display_Measures_12_04_2025.csv'
REPORT_CARD        = CONTENT / '4. 2026_Report_Card_Master_Table_2025_10_08.xlsx'

print('File paths configured.')
for p in [ACHP_XWALK_2026, ACHP_XWALK_2025, LANDSCAPE_2026, LANDSCAPE_2025,
          DISPLAY_MEASURES, REPORT_CARD]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p.name}')


File paths configured.
  ✗ MISSING  1. ACHP_Members_Crosswalk_2026_JHU (List of ACHP Member Plans).xlsx
  ✗ MISSING  1. ACHP_Members_Crosswalk_2025_JHU (List of ACHP Member Plans).xlsx
  ✓  2. CY2026_Landscape_202603.csv
  ✓  5. CY2025_Landscape_202506.1.csv
  ✗ MISSING  4. 2026_HPMS_Display_Measures_12_04_2025.csv
  ✗ MISSING  4. 2026_Report_Card_Master_Table_2025_10_08.xlsx


### Step 1 — Aggregate CPSC Monthly Enrollment (2025)

CMS publishes MA enrollment as 12 separate monthly files.
We aggregate all 12 months into a single annual file, summing enrollment
by contract, plan, state, and county. CMS suppresses values of 10 or fewer
enrollees with `*`; we treat those as zero to preserve row coverage.


In [ ]:
# We read all 12 monthly enrollment ZIPs and stack them into one DataFrame.
# Enrollment is reported monthly by CMS, so we sum across months to get
# an annual figure for each plan-county combination.

all_months = []
for i in range(1, 13):
    zip_path = CPSC_DIR / f'cpsc_2025_{i:02d}.zip'
    csv_name = f'CPSC_Enrollment_2025_{i:02d}/CPSC_Enrollment_Info_2025_{i:02d}.csv'
    with zipfile.ZipFile(zip_path) as z:
        with z.open(csv_name) as f_csv:
            all_months.append(pd.read_csv(f_csv, encoding='latin1'))
    print(f'  Loaded month {i:02d}')

combined = pd.concat(all_months, ignore_index=True)

# CMS uses '*' to mask small counts — we treat these as 0
combined['Enrollment'] = (
    combined['Enrollment'].astype(str).str.strip()
    .replace('*', '0')
    .pipe(pd.to_numeric, errors='coerce')
    .fillna(0)
)
for col in ['Contract Number', 'Plan ID', 'State', 'County']:
    combined[col] = combined[col].astype(str).str.strip()

enrollment_annual = (
    combined
    .groupby(['Contract Number', 'Plan ID', 'State', 'County'], as_index=False)['Enrollment']
    .sum()
    .rename(columns={'Contract Number': 'Contract ID'})
)

enrollment_annual.to_csv(CONTENT / 'cpsc_enrollment_2025_aggregated.csv', index=False)
print(f'\nAggregated enrollment: {len(enrollment_annual):,} plan-county rows')
print(f'Total enrollees across all plans: {enrollment_annual["Enrollment"].sum():,.0f}')


### Step 2 — Load CMS Star Ratings & Performance Data (2026)

We pull four performance data tables from CMS — display measures, summary
star ratings, per-measure star scores, and raw measure data — and join them
to the landscape file on Contract ID. This gives us quality context alongside
the financial attributes we use in our analysis.


In [ ]:
def load_performance_data():
    # Display measures — wide format with a 3-row header we need to skip
    raw_disp = pd.read_csv(DISPLAY_MEASURES, encoding='latin1', header=None)
    disp_cols = raw_disp.iloc[2].tolist()
    disp_cols[:4] = ['Contract Number', 'Organization Marketing Name',
                     'Contract Name', 'Parent Organization']
    display = raw_disp.iloc[3:].reset_index(drop=True)
    display.columns = disp_cols
    display['Contract Number'] = display['Contract Number'].astype(str).str.strip()
    display = display.drop(columns=['Organization Marketing Name',
                                    'Contract Name', 'Parent Organization'])
    display = (display.add_prefix('Display_')
                      .rename(columns={'Display_Contract Number': 'Contract ID'}))

    def _load_report_sheet(sheet_name, prefix):
        raw = pd.read_excel(REPORT_CARD, sheet_name=sheet_name, header=None)
        cols = raw.iloc[2].tolist()
        cols[:5] = ['Contract ID', 'Organization Type', 'Contract Name',
                    'Organization Marketing Name', 'Parent Organization']
        df = raw.iloc[4:].reset_index(drop=True)
        df.columns = cols
        df['Contract ID'] = df['Contract ID'].astype(str).str.strip()
        df = df.drop(columns=['Organization Type', 'Contract Name',
                              'Organization Marketing Name', 'Parent Organization'],
                     errors='ignore')
        return df.add_prefix(f'{prefix}_').rename(columns={f'{prefix}_Contract ID': 'Contract ID'})

    stars   = _load_report_sheet('Measure_Stars',   'MeasureStars')
    mdata   = _load_report_sheet('Measure_Data',    'MeasureData')

    # Summary ratings use a 2-row header
    raw_sum = pd.read_excel(REPORT_CARD, sheet_name='Summary_Rating', header=None)
    sum_cols = raw_sum.iloc[1].tolist(); sum_cols[0] = 'Contract ID'
    summary = raw_sum.iloc[2:].reset_index(drop=True)
    summary.columns = sum_cols
    summary['Contract ID'] = summary['Contract ID'].astype(str).str.strip()
    summary = summary.drop(columns=['Organization Type', 'Contract Name',
                                    'Organization Marketing Name', 'Parent Organization'],
                           errors='ignore')
    summary = (summary.add_prefix('SummaryRating_')
                      .rename(columns={'SummaryRating_Contract ID': 'Contract ID'}))

    return display, summary, stars, mdata

print('Performance data loader defined.')


### Step 3 — Build 2026 Master File

We join four sources on Contract ID and the four-column geographic key
(Contract ID, Plan ID, State, County) to create our 2026 analytical master file:

1. **CMS Landscape 2026** — base plan attributes, premiums, cost-sharing
2. **ACHP crosswalk** — flags each contract as ACHP member (1) or non-member (0)
3. **CPSC enrollment** — annual enrollment per plan-county
4. **CMS performance data** — star ratings and quality measures


In [ ]:
# Load the ACHP member crosswalk and flag every contract in the landscape file.
# This is our primary cohort-construction step: ACHP vs. non-ACHP.

print('Loading 2026 landscape and ACHP crosswalk...')
achp_2026    = pd.read_excel(ACHP_XWALK_2026)
landscape_26 = pd.read_csv(LANDSCAPE_2026, low_memory=False)

achp_ids_26 = set(achp_2026['Contract ID'].astype(str).unique())
landscape_26['ACHP?'] = landscape_26['Contract ID'].astype(str).apply(
    lambda x: 1 if x in achp_ids_26 else 0
)
print(f'  ACHP plans:     {(landscape_26["ACHP?"]==1).sum():,}')
print(f'  Non-ACHP plans: {(landscape_26["ACHP?"]==0).sum():,}')

# Join enrollment — we match on the full four-column geographic key
# to preserve one row per plan-county, which is our unit of analysis.
print('\nJoining enrollment data...')
enrl = pd.read_csv(CONTENT / 'cpsc_enrollment_2025_aggregated.csv')
merged_26 = landscape_26.merge(
    enrl[['Contract ID', 'Plan ID', 'State', 'County', 'Enrollment']],
    left_on=['Contract ID', 'Plan ID', 'State Territory Abbreviation', 'County Name'],
    right_on=['Contract ID', 'Plan ID', 'State', 'County'],
    how='left'
).drop(columns=['State', 'County'])
print(f'  Matched rows:   {merged_26["Enrollment"].notna().sum():,}')
print(f'  Unmatched rows: {merged_26["Enrollment"].isna().sum():,}')

# Add star ratings and performance measures from CMS report card
print('\nJoining CMS performance data...')
display, summary, stars, mdata = load_performance_data()
master_2026 = (merged_26
               .merge(display,  on='Contract ID', how='left')
               .merge(summary,  on='Contract ID', how='left')
               .merge(stars,    on='Contract ID', how='left')
               .merge(mdata,    on='Contract ID', how='left'))

master_2026.to_csv(CONTENT / '2026_master.csv', index=False)
print(f'\n✓ 2026_master.csv saved — {master_2026.shape[0]:,} rows × {master_2026.shape[1]} columns')


### Step 4 — Build 2025 Master File

We apply the same joining logic to the 2025 landscape file.
Note that the 2025 landscape uses `State Abbreviation` (not `State Territory Abbreviation`),
so we adjust the join key accordingly.


In [ ]:
print('Loading 2025 landscape and ACHP crosswalk...')
achp_2025    = pd.read_excel(ACHP_XWALK_2025)
landscape_25 = pd.read_csv(LANDSCAPE_2025, low_memory=False)

achp_ids_25 = set(achp_2025['Contract ID'].astype(str).unique())
landscape_25['ACHP?'] = landscape_25['Contract ID'].astype(str).apply(
    lambda x: 1 if x in achp_ids_25 else 0
)
print(f'  ACHP plans:     {(landscape_25["ACHP?"]==1).sum():,}')
print(f'  Non-ACHP plans: {(landscape_25["ACHP?"]==0).sum():,}')

# 2025 landscape uses 'State Abbreviation' rather than 'State Territory Abbreviation'
print('\nJoining enrollment data...')
merged_25 = landscape_25.merge(
    enrl[['Contract ID', 'Plan ID', 'State', 'County', 'Enrollment']],
    left_on=['Contract ID', 'Plan ID', 'State Abbreviation', 'County Name'],
    right_on=['Contract ID', 'Plan ID', 'State', 'County'],
    how='left'
).drop(columns=['State', 'County'])
print(f'  Matched rows:   {merged_25["Enrollment"].notna().sum():,}')
print(f'  Unmatched rows: {merged_25["Enrollment"].isna().sum():,}')

print('\nJoining CMS performance data...')
master_2025 = (merged_25
               .merge(display,  on='Contract ID', how='left')
               .merge(summary,  on='Contract ID', how='left')
               .merge(stars,    on='Contract ID', how='left')
               .merge(mdata,    on='Contract ID', how='left'))

master_2025.to_csv(CONTENT / '2025_master.csv', index=False)
print(f'\n✓ 2025_master.csv saved — {master_2025.shape[0]:,} rows × {master_2025.shape[1]} columns')


### Step 5 — Verify Master Files

A quick sanity check before proceeding to analysis:
we confirm row counts, ACHP coverage, and enrollment match rates for both years.


In [ ]:
df25_check = pd.read_csv(CONTENT / '2025_master.csv', low_memory=False)
df26_check = pd.read_csv(CONTENT / '2026_master.csv', low_memory=False)

print('Master file summary:')
print(f'  2025: {df25_check.shape[0]:>7,} rows  |  {df25_check.shape[1]} columns  '
      f'|  ACHP plans: {(df25_check["ACHP?"]==1).sum():,}  '
      f'|  Enrollment matched: {df25_check["Enrollment"].notna().sum():,}')
print(f'  2026: {df26_check.shape[0]:>7,} rows  |  {df26_check.shape[1]} columns  '
      f'|  ACHP plans: {(df26_check["ACHP?"]==1).sum():,}  '
      f'|  Enrollment matched: {df26_check["Enrollment"].notna().sum():,}')

# Spot-check: confirm star ratings joined correctly
star_col = 'SummaryRating_2026 Overall'
if star_col in df26_check.columns:
    rated = df26_check[star_col].notna().sum()
    print(f'\n  2026 plans with overall star rating: {rated:,} '
          f'({rated/len(df26_check)*100:.1f}%)')

print('\nReady to proceed to the year-over-year analysis pipeline.')


---
## Analysis Pipeline — Year-over-Year Benchmarking

The cells below load the master files constructed above and run our
full year-over-year analysis: YoY merge, EDA validation, and Findings 1–3.


# Group 12 — ACHP Medicare Advantage Analysis
## Full Pipeline: Merge → EDA Validation → Findings 1, 2, 3
### Fixes applied per professor critique:
- ✅ No deduplication — all plan-county rows preserved
- ✅ Enrollment-weighted star ratings
- ✅ KP isolated as separate cohort
- ✅ All clusters shown in Finding 3

**How to use:**
1. Upload `2025_master.csv` and `2026_master.csv` to `/content/`
2. Upload your two raw CMS Landscape CSVs to `/content/`
3. Update the landscape filenames in Cell 3 if needed
4. Run all cells top to bottom


## Cell 1 — Install & Imports

In [ ]:
!pip install matplotlib seaborn statsmodels openpyxl scikit-learn -q

import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.stats import chi2_contingency
from openpyxl.drawing.image import Image as XLImage
warnings.filterwarnings('ignore')

OUT = '/content/'
print('✓ Libraries ready')


✓ Libraries ready


## Cell 2 — Load 2025 & 2026 Master Files

In [2]:
PATH_25 = '/content/2025_master.csv'
PATH_26 = '/content/2026_master.csv'

print('Loading 2026 master...')
df26 = pd.read_csv(PATH_26, low_memory=False)
print(f'  2026: {df26.shape[0]:,} rows x {df26.shape[1]} cols')

print('Loading 2025 master...')
df25 = pd.read_csv(PATH_25, low_memory=False)
print(f'  2025: {df25.shape[0]:,} rows x {df25.shape[1]} cols')

# 2026 uses "State Territory Abbreviation", 2025 uses "State Abbreviation"
df26 = df26.rename(columns={
    'State Territory Abbreviation': 'State Abbreviation',
    'State Territory Name':         'State Name',
})

# Ensure ContractPlanID exists
for df, label in [(df25,'2025'),(df26,'2026')]:
    if 'ContractPlanID' not in df.columns:
        df['ContractPlanID'] = (df['Contract ID'].astype(str).str.strip() + '_' +
                                df['Plan ID'].astype(str).str.strip().str.zfill(3))

print(f'\n  State col in 2025: {"State Abbreviation" in df25.columns}')
print(f'  State col in 2026: {"State Abbreviation" in df26.columns}')
print(f'  2025 ACHP rows: {(df25["ACHP?"]==1).sum():,}')
print(f'  2026 ACHP rows: {(df26["ACHP?"]==1).sum():,}')
print('\n✓ Master files loaded')


Loading 2026 master...
  2026: 138,263 rows x 208 cols
Loading 2025 master...
  2025: 142,653 rows x 204 cols

  State col in 2025: True
  State col in 2026: True
  2025 ACHP rows: 8,202
  2026 ACHP rows: 7,337

✓ Master files loaded


## Cell 3 — Patch Financial Columns from Raw Landscape Files
Update the two filenames below to match what you uploaded to `/content/`.


In [6]:
LANDSCAPE_25 = '/content/5. CY2025_Landscape_202506.1.csv'  # <- update if needed
LANDSCAPE_26 = '/content/2. CY2026_Landscape_202603.csv'   # <- update if needed

FINANCIAL_COLS = [
    'Monthly Consolidated Premium (Part C + D)',
    'In-Network Maximum Out-of-Pocket (MOOP) Amount',
    'Annual Part D Deductible Amount',
    'Part D Basic Premium',
    'Part D Total Premium',
    'Part C Premium',
    'Part D Supplemental Premium',
]
JOIN_KEY = ['Contract ID', 'Plan ID', 'State Abbreviation', 'County Name']

def clean_currency(series):
    return (series.astype(str)
                  .str.replace(r'[$,\s]', '', regex=True)
                  .replace('NotApplicable', float('nan'))
                  .pipe(pd.to_numeric, errors='coerce'))

def load_landscape(path, state_col):
    print(f'Loading {path}...')
    raw = pd.read_csv(path, low_memory=False)
    print(f'  Shape: {raw.shape}')
    if state_col in raw.columns and state_col != 'State Abbreviation':
        raw = raw.rename(columns={state_col: 'State Abbreviation'})
    for col in FINANCIAL_COLS:
        if col in raw.columns:
            raw[col] = clean_currency(raw[col])
    for col in JOIN_KEY:
        if col in raw.columns:
            raw[col] = raw[col].astype(str).str.strip()
    nn = raw['Monthly Consolidated Premium (Part C + D)'].notna().sum()
    print(f'  Premium non-null after cleaning: {nn:,} / {len(raw):,}')
    print(f'  Sample: {raw["Monthly Consolidated Premium (Part C + D)"].dropna().head(3).tolist()}')
    return raw

land25 = load_landscape(LANDSCAPE_25, 'State Abbreviation')
land26 = load_landscape(LANDSCAPE_26, 'State Territory Abbreviation')

def patch_master(df, land, label):
    fin = [c for c in FINANCIAL_COLS if c in land.columns]
    df  = df.drop(columns=[c for c in fin if c in df.columns], errors='ignore')
    for col in JOIN_KEY:
        df[col]   = df[col].astype(str).str.strip()
        land[col] = land[col].astype(str).str.strip()
    df = df.merge(land[JOIN_KEY + fin], on=JOIN_KEY, how='left')
    nn = df['Monthly Consolidated Premium (Part C + D)'].notna().sum()
    print(f'[{label}] Premium non-null after patch: {nn:,} / {len(df):,}')
    return df

df25 = patch_master(df25, land25, '2025')
df26 = patch_master(df26, land26, '2026')
print('\n✓ Financial columns patched — re-run from Cell 4 onwards')


Loading /content/5. CY2025_Landscape_202506.1.csv...
  Shape: (142653, 49)
  Premium non-null after cleaning: 119,264 / 142,653
  Sample: [29.5, 153.0, 0.0]
Loading /content/2. CY2026_Landscape_202603.csv...
  Shape: (138263, 53)
  Premium non-null after cleaning: 116,545 / 138,263
  Sample: [35.5, 163.0, 0.0]
[2025] Premium non-null after patch: 119,264 / 142,653
[2026] Premium non-null after patch: 116,545 / 138,263

✓ Financial columns patched — re-run from Cell 4 onwards


## Cell 4 — Year-over-Year Merge (2025 → 2026)
Join on full 4-column key. **No deduplication applied.**


In [7]:
MERGE_KEY = ['Contract ID', 'Plan ID', 'State Abbreviation', 'County Name']

CARRY = ['ContractPlanID',
         'Monthly Consolidated Premium (Part C + D)',
         'In-Network Maximum Out-of-Pocket (MOOP) Amount',
         'Annual Part D Deductible Amount',
         'Part D Basic Premium', 'Part C Premium',
         'Overall Star Rating', 'Enrollment',
         'ACHP?', 'Plan Type', 'SNP Type',
         'Organization Type', 'Organization Marketing Name',
         'Parent Organization Name', 'Contract Name', 'MA Region',
         'SummaryRating_2026 Overall',
]

def safe_carry(df, cols, key):
    avail = key + [c for c in cols if c in df.columns and c not in key]
    return df[avail].copy()

# Strip whitespace on keys before merge
for df in [df25, df26]:
    for col in MERGE_KEY:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

left  = safe_carry(df25, CARRY, MERGE_KEY)
right = safe_carry(df26, CARRY, MERGE_KEY)

yoy = left.merge(right, on=MERGE_KEY, how='inner', suffixes=('_2025','_2026'))

print(f'Matched plan-county rows: {len(yoy):,}')
print(f'  Unmatched 2025 rows:    {len(left)  - len(yoy):,}')
print(f'  Unmatched 2026 rows:    {len(right) - len(yoy):,}')

# YoY deltas
yoy['Delta_Consolidated_Premium'] = (
    yoy['Monthly Consolidated Premium (Part C + D)_2026'] -
    yoy['Monthly Consolidated Premium (Part C + D)_2025'])
yoy['Delta_MOOP'] = (
    yoy['In-Network Maximum Out-of-Pocket (MOOP) Amount_2026'] -
    yoy['In-Network Maximum Out-of-Pocket (MOOP) Amount_2025'])
yoy['Delta_PartD_Deductible'] = (
    yoy['Annual Part D Deductible Amount_2026'] -
    yoy['Annual Part D Deductible Amount_2025'])

# Kaiser Permanente flag
def is_kp(df, col):
    if col in df.columns:
        return df[col].fillna('').str.upper().str.contains('KAISER')
    return pd.Series(False, index=df.index)

kp_mask = (is_kp(yoy,'Organization Marketing Name_2026') |
           is_kp(yoy,'Parent Organization Name_2026') |
           is_kp(yoy,'Contract Name_2026'))
yoy['KP_Flag'] = kp_mask.astype(int)

# Cohort assignment
achp_col  = 'ACHP?_2026' if 'ACHP?_2026' in yoy.columns else 'ACHP?_2025'
achp_mask = yoy[achp_col] == 1
yoy['Cohort']   = 'National'
yoy.loc[achp_mask & (yoy['KP_Flag']==0), 'Cohort'] = 'ACHP-nonKP'
yoy.loc[achp_mask & (yoy['KP_Flag']==1), 'Cohort'] = 'ACHP-KP'
yoy['ACHP_All'] = achp_mask.astype(int)

print('\nCohort distribution:')
print(yoy['Cohort'].value_counts().to_string())
print(f'\nDelta premium non-null:    {yoy["Delta_Consolidated_Premium"].notna().sum():,}')
print(f'Delta MOOP non-null:       {yoy["Delta_MOOP"].notna().sum():,}')
print(f'Delta deductible non-null: {yoy["Delta_PartD_Deductible"].notna().sum():,}')

# ── Enrollment /12 fix: CMS reports monthly enrollment; divide to get annual ──
for _ecol in ['Enrollment_2025', 'Enrollment_2026']:
    if _ecol in yoy.columns:
        yoy[_ecol] = pd.to_numeric(yoy[_ecol], errors='coerce') / 12


Matched plan-county rows: 109,015
  Unmatched 2025 rows:    33,638
  Unmatched 2026 rows:    29,248

Cohort distribution:
Cohort
National      102450
ACHP-nonKP      6160
ACHP-KP          405

Delta premium non-null:    90,730
Delta MOOP non-null:       108,325
Delta deductible non-null: 91,391


## Cell 5 — EDA Validation

In [8]:
def check_col(df, col, label):
    if col not in df.columns:
        return {'Column': col, 'Label': label, 'Non-null': 0, 'Coverage': '0%', 'Status': 'MISSING'}
    nn  = df[col].notna().sum()
    pct = nn / len(df) * 100
    status = 'OK' if pct > 50 else ('LOW' if pct > 5 else 'CRITICAL')
    return {'Column': col, 'Label': label, 'Non-null': nn,
            'Coverage': f'{pct:.1f}%', 'Status': status}

checks = []
for col, lbl in [
    ('Delta_Consolidated_Premium',                           'F1: Delta Premium'),
    ('Delta_MOOP',                                           'F1: Delta MOOP'),
    ('Delta_PartD_Deductible',                               'F1: Delta Deductible'),
    ('Enrollment_2025',                                      'F1: 2025 Enrollment'),
    ('Overall Star Rating_2025',                             'F1: Star Rating'),
    ('Monthly Consolidated Premium (Part C + D)_2025',       'F1: 2025 Premium'),
    ('Monthly Consolidated Premium (Part C + D)_2026',       'F1: 2026 Premium'),
    ('Annual Part D Deductible Amount_2025',                 'F2: 2025 Deductible'),
    ('Annual Part D Deductible Amount_2026',                 'F2: 2026 Deductible'),
    ('Monthly Consolidated Premium (Part C + D)_2026',       'F3: Premium'),
    ('In-Network Maximum Out-of-Pocket (MOOP) Amount_2026',  'F3: MOOP'),
    ('Overall Star Rating_2026',                             'F3: Stars'),
    ('Enrollment_2026',                                      'F3: Enrollment'),
    ('Plan Type_2026',                                       'F3: Plan Type'),
    ('SNP Type_2026',                                        'F3: SNP Type'),
]:
    checks.append(check_col(yoy, col, lbl))

eda_df = pd.DataFrame(checks)
print(eda_df.to_string(index=False))
critical = eda_df[eda_df['Status']=='CRITICAL']
if len(critical)==0:
    print('\nALL CHECKS PASSED')
else:
    print(f'\n{len(critical)} critical issues found')
    print(critical[['Column','Coverage']].to_string(index=False))


                                             Column                Label  Non-null Coverage Status
                         Delta_Consolidated_Premium    F1: Delta Premium     90730    83.2%     OK
                                         Delta_MOOP       F1: Delta MOOP    108325    99.4%     OK
                             Delta_PartD_Deductible F1: Delta Deductible     91391    83.8%     OK
                                    Enrollment_2025  F1: 2025 Enrollment    108385    99.4%     OK
                           Overall Star Rating_2025      F1: Star Rating    109015   100.0%     OK
     Monthly Consolidated Premium (Part C + D)_2025     F1: 2025 Premium     90730    83.2%     OK
     Monthly Consolidated Premium (Part C + D)_2026     F1: 2026 Premium     90730    83.2%     OK
               Annual Part D Deductible Amount_2025  F2: 2025 Deductible     91391    83.8%     OK
               Annual Part D Deductible Amount_2026  F2: 2026 Deductible     91391    83.8%     OK
     Month

## Cell 6 — Finding 1 Setup: Filter MA Plans, Exclude SNP
Uses Organization Type values `Local CCP` and `Regional CCP` to identify MA plans.


In [9]:
org_col = 'Organization Type_2026' if 'Organization Type_2026' in yoy.columns else 'Organization Type_2025'
snp_col = 'SNP Type_2026'          if 'SNP Type_2026'          in yoy.columns else 'SNP Type_2025'

ma_mask  = yoy[org_col].isin(['Local CCP', 'Regional CCP'])
snp_mask = yoy[snp_col].astype(str).str.strip() == 'Not Applicable'
f1       = yoy[ma_mask & snp_mask].copy()

print(f'Total yoy rows:      {len(yoy):,}')
print(f'After MA filter:     {ma_mask.sum():,}')
print(f'After SNP exclusion: {len(f1):,}')
print('\nCohort breakdown:')
print(f1['Cohort'].value_counts().to_string())

COHORT_ORDER  = ['ACHP-KP', 'ACHP-nonKP', 'National']
COHORT_COLORS = {'ACHP-KP': '#27ae60', 'ACHP-nonKP': '#2ecc71', 'National': '#e74c3c'}

def weighted_mean(series, weights):
    mask = series.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0: return np.nan
    return np.average(series[mask], weights=weights[mask])

charts_f1 = {}
print('\n✓ F1 dataset ready')


Total yoy rows:      109,015
After MA filter:     105,900
After SNP exclusion: 68,165

Cohort breakdown:
Cohort
National      63918
ACHP-nonKP     3906
ACHP-KP         341

✓ F1 dataset ready


## Cell 7 — Finding 1: Chart 1 — Grouped Bar (Weighted Δ Premium & MOOP)

In [10]:
# ── Color scheme: gold for ACHP cohorts, grey for National ──
GOLD   = '#D4A017'
GREY   = '#A8A8A8'
VIZ_COHORT_COLORS = {
    'ACHP-KP':    GOLD,
    'ACHP-nonKP': GOLD,
    'National':   GREY,
    'Regional':   '#C8C8C8',
}

w_col    = 'Enrollment_2025'
bar_rows = []
for cohort in COHORT_ORDER:
    sub = f1[f1['Cohort']==cohort].copy()
    w   = sub[w_col] if w_col in sub.columns else pd.Series(1, index=sub.index)

    star_col = 'Overall Star Rating_2026' if 'Overall Star Rating_2026' in sub.columns else \
               'Overall Star Rating_2025' if 'Overall Star Rating_2025' in sub.columns else None
    wtd_stars = weighted_mean(pd.to_numeric(sub[star_col], errors='coerce'), w) \
                if star_col else np.nan

    bar_rows.append({
        'Cohort':           cohort,
        'Delta_Premium':    weighted_mean(sub['Delta_Consolidated_Premium'], w),
        'Delta_MOOP':       weighted_mean(sub['Delta_MOOP'], w),
        'Delta_Deductible': weighted_mean(sub['Delta_PartD_Deductible'], w),
        'Plan_Count':       len(sub),
        'Total_Enrollment': int(w.sum()) if w_col in sub.columns else len(sub),
        'Wtd_Stars':        wtd_stars,
    })

bar_df          = pd.DataFrame(bar_rows)
market_avg_prem = weighted_mean(f1['Delta_Consolidated_Premium'], f1[w_col])
market_avg_moop = weighted_mean(f1['Delta_MOOP'], f1[w_col])

print('Enrollment-weighted cohort stats:')
print(bar_df.to_string(index=False))
print(f'\nMarket avg Delta premium: ${market_avg_prem:.2f}')
print(f'Market avg Delta MOOP:    ${market_avg_moop:.2f}')
print('(Stars are enrollment-weighted — addresses professor question)')

fig, axes = plt.subplots(1, 2, figsize=(14,6))
fig.suptitle('Enrollment-Weighted YoY Cost Changes by Cohort (MA, SNP Excluded)\n'
             '2025 -> 2026  |  KP shown separately  |  Stars are enrollment-weighted',
             fontsize=13, fontweight='bold', y=1.02)

for ax, col, ref, title in [
    (axes[0], 'Delta_Premium', market_avg_prem, 'Consolidated Premium Change ($)'),
    (axes[1], 'Delta_MOOP',   market_avg_moop,  'MOOP Change ($)'),
]:
    vals   = bar_df[col]
    colors = [VIZ_COHORT_COLORS.get(c, GREY) for c in bar_df['Cohort']]
    bars   = ax.bar(bar_df['Cohort'], vals, color=colors, width=0.5, edgecolor='white')
    for bar, v in zip(bars, vals):
        if pd.notna(v):
            ypos = bar.get_height() + 0.3 if v >= 0 else bar.get_height() - 0.8
            ax.text(bar.get_x()+bar.get_width()/2, ypos, f'${v:+.1f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.axhline(ref, color='#2c3e50', linewidth=1.5, linestyle='--', label=f'Market avg ${ref:+.1f}')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Weighted Avg Change ($)', fontsize=10)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
buf1 = io.BytesIO()
plt.savefig(buf1, format='png', dpi=150, bbox_inches='tight'); buf1.seek(0)
charts_f1['chart1_grouped_bar'] = buf1
plt.savefig(f'{OUT}chart1_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('chart1_grouped_bar.png saved')


Enrollment-weighted cohort stats:
    Cohort  Delta_Premium  Delta_MOOP  Delta_Deductible  Plan_Count  Total_Enrollment  Wtd_Stars
   ACHP-KP      10.474240  226.007327         -0.487603         341           1155733   4.455092
ACHP-nonKP       4.620671  331.665201         96.571428        3906           1034921   4.112879
  National       2.761911  311.252686        160.933324       63918          15512946   3.871609

Market avg Delta premium: $3.39
Market avg Delta MOOP:    $306.88
(Stars are enrollment-weighted — addresses professor question)
chart1_grouped_bar.png saved


## Cell 8 — Finding 1: Chart 2 — Diverging Distribution

In [11]:
GOLD = '#D4A017'; GREY = '#A8A8A8'
DIST_COLORS = {
    'ACHP-KP':    GOLD,
    'ACHP-nonKP': GOLD,
    'National':   GREY,
    'Regional':   '#C8C8C8',
}

dist_df = f1[f1['Delta_Consolidated_Premium'].notna()].copy()
clip_hi = dist_df['Delta_Consolidated_Premium'].quantile(0.99)
clip_lo = dist_df['Delta_Consolidated_Premium'].quantile(0.01)
dist_df['Delta_clipped'] = dist_df['Delta_Consolidated_Premium'].clip(clip_lo, clip_hi)

fig, axes = plt.subplots(len(COHORT_ORDER), 1, figsize=(12,10), sharex=True)
fig.suptitle('Distribution of Plan-Level Premium Changes (2025->2026)\nMA Plans, SNP Excluded | KP separate',
             fontsize=13, fontweight='bold')

for ax, cohort in zip(axes, COHORT_ORDER):
    sub    = dist_df[dist_df['Cohort']==cohort]['Delta_clipped']
    color  = DIST_COLORS.get(cohort, GREY)
    n      = len(sub)
    pct_up = (sub > 0).mean()*100 if n > 0 else 0
    pct_dn = (sub < 0).mean()*100 if n > 0 else 0
    pct_fl = (sub == 0).mean()*100 if n > 0 else 0
    raise_color = GOLD if 'ACHP' in cohort else '#888888'
    cut_color   = '#D0D0D0' if 'ACHP' in cohort else '#C0C0C0'
    ax.hist(sub[sub<=0], bins=40, color=cut_color, alpha=0.85, label='Cut / Flat')
    ax.hist(sub[sub>0],  bins=40, color=raise_color, alpha=0.85, label='Raised')
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    if n > 0:
        ax.axvline(sub.mean(), color=color, linewidth=1.5, label=f'Mean ${sub.mean():+.1f}')
    ax.set_title(f'{cohort}  (n={n:,})   Raised:{pct_up:.1f}%  Flat:{pct_fl:.1f}%  Cut:{pct_dn:.1f}%',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Plan Count', fontsize=9)
    ax.legend(fontsize=9, loc='upper right')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('Consolidated Premium Change ($)', fontsize=11)
plt.tight_layout()
buf2 = io.BytesIO()
plt.savefig(buf2, format='png', dpi=150, bbox_inches='tight'); buf2.seek(0)
charts_f1['chart2_diverging_dist'] = buf2
plt.savefig(f'{OUT}chart2_diverging_dist.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('chart2_diverging_dist.png saved')


chart2_diverging_dist.png saved


## Cell 9 — Finding 1: Chart 3 — 2025 vs 2026 Premium Scatter

In [12]:
GOLD = '#D4A017'; GREY = '#A8A8A8'
SCATTER_COLORS = {
    'ACHP-KP':    GOLD,
    'ACHP-nonKP': GOLD,
    'National':   GREY,
    'Regional':   '#C8C8C8',
}

p25 = 'Monthly Consolidated Premium (Part C + D)_2025'
p26 = 'Monthly Consolidated Premium (Part C + D)_2026'

sc_df = f1[[p25, p26, 'Cohort']].dropna()
sc_df = sc_df[(sc_df[p25] < sc_df[p25].quantile(0.99)) &
              (sc_df[p26] < sc_df[p26].quantile(0.99))]

fig, ax = plt.subplots(figsize=(10,7))
for cohort in ['National', 'Regional', 'ACHP-nonKP', 'ACHP-KP']:
    if cohort not in sc_df['Cohort'].values:
        continue
    sub   = sc_df[sc_df['Cohort']==cohort]
    color = SCATTER_COLORS.get(cohort, GREY)
    alpha = 0.2 if cohort in ('National','Regional') else 0.55
    size  = 12  if cohort in ('National','Regional') else 22
    ax.scatter(sub[p25], sub[p26], color=color, alpha=alpha, s=size, label=cohort)

lo = min(sc_df[p25].min(), sc_df[p26].min())
hi = max(sc_df[p25].max(), sc_df[p26].max())
ax.plot([lo,hi],[lo,hi],'k--',linewidth=1, label='No change')
ax.set_xlabel('2025 Consolidated Premium ($)', fontsize=11)
ax.set_ylabel('2026 Consolidated Premium ($)', fontsize=11)
ax.set_title('2025 vs 2026 Premium by Cohort\n(above diagonal = premium increase)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
buf3 = io.BytesIO()
plt.savefig(buf3, format='png', dpi=150, bbox_inches='tight'); buf3.seek(0)
charts_f1['chart3_scatter'] = buf3
plt.savefig(f'{OUT}chart3_scatter.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('chart3_scatter.png saved')


chart3_scatter.png saved


## Cell 10 — Finding 1: OLS Regression

In [13]:
reg_df = f1.copy()
reg_df['ACHP_Flag']      = (reg_df['Cohort'].isin(['ACHP-KP','ACHP-nonKP'])).astype(int)
reg_df['Log_Enrollment'] = np.log1p(pd.to_numeric(
    reg_df.get('Enrollment_2025', pd.Series(0, index=reg_df.index)), errors='coerce').fillna(0))

for star_col in ['Overall Star Rating_2025','Overall Star Rating_2026',
                 'Overall Star Rating','SummaryRating_2026 Overall']:
    if star_col in reg_df.columns:
        reg_df['Star_Rating_Num'] = pd.to_numeric(reg_df[star_col], errors='coerce')
        print(f'Star column: {star_col} — non-null: {reg_df["Star_Rating_Num"].notna().sum():,}')
        break
else:
    reg_df['Star_Rating_Num'] = 0.0
    print('No star column found — using 0')

reg_df['Star_Rating_Num'] = reg_df['Star_Rating_Num'].fillna(0)
for col in ['Delta_Consolidated_Premium','Delta_MOOP']:
    reg_df[col] = pd.to_numeric(reg_df[col], errors='coerce')

required  = ['Delta_Consolidated_Premium','Delta_MOOP','ACHP_Flag','Log_Enrollment']
reg_clean = reg_df[required + ['Star_Rating_Num']].dropna(subset=required)

print(f'\nRegression sample size: {len(reg_clean):,}')
print(f'ACHP: {reg_clean["ACHP_Flag"].sum():,} | National: {(reg_clean["ACHP_Flag"]==0).sum():,}')

model_premium = smf.ols('Delta_Consolidated_Premium ~ ACHP_Flag + Star_Rating_Num + Log_Enrollment',
                         data=reg_clean).fit(cov_type='HC3')
model_moop    = smf.ols('Delta_MOOP ~ ACHP_Flag + Star_Rating_Num + Log_Enrollment',
                         data=reg_clean).fit(cov_type='HC3')

achp_coef_p = model_premium.params['ACHP_Flag']
achp_pval_p = model_premium.pvalues['ACHP_Flag']
achp_ci_p   = model_premium.conf_int().loc['ACHP_Flag'].values
achp_coef_m = model_moop.params['ACHP_Flag']
achp_pval_m = model_moop.pvalues['ACHP_Flag']
achp_ci_m   = model_moop.conf_int().loc['ACHP_Flag'].values

summary_p_df = pd.DataFrame([{
    'Outcome':'Delta_Premium', 'ACHP_Coef':round(achp_coef_p,3),
    'ACHP_pval':round(achp_pval_p,4), 'CI_low':round(achp_ci_p[0],3),
    'CI_high':round(achp_ci_p[1],3), 'Sig':achp_pval_p<0.05,
    'R2':round(model_premium.rsquared,4),
},{
    'Outcome':'Delta_MOOP', 'ACHP_Coef':round(achp_coef_m,3),
    'ACHP_pval':round(achp_pval_m,4), 'CI_low':round(achp_ci_m[0],3),
    'CI_high':round(achp_ci_m[1],3), 'Sig':achp_pval_m<0.05,
    'R2':round(model_moop.rsquared,4),
}])
print('\n=== OLS Key Results ===')
print(summary_p_df.to_string(index=False))


Star column: Overall Star Rating_2025 — non-null: 66,078

Regression sample size: 51,918
ACHP: 3,399 | National: 48,519

=== OLS Key Results ===
      Outcome  ACHP_Coef  ACHP_pval  CI_low  CI_high  Sig     R2
Delta_Premium      2.810        0.0   2.518    3.102 True 0.0210
   Delta_MOOP     96.673        0.0  72.901  120.445 True 0.0016


## Cell 11 — Finding 1: Chart 4 — OLS Coefficient Plot

In [14]:
# Gold for ACHP_Flag (focal), grey for controls
GOLD = '#D4A017'; GREY = '#A8A8A8'

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('OLS Regression Coefficients (HC3 robust SE)\nMA Plans, SNP Excluded',
             fontsize=13, fontweight='bold')

for ax, model, title in [
    (axes[0], model_premium, 'Outcome: Delta Consolidated Premium'),
    (axes[1], model_moop,    'Outcome: Delta MOOP'),
]:
    show  = [s for s in ['ACHP_Flag','Star_Rating_Num','Log_Enrollment'] if s in model.params.index]
    coefs = model.params[show]
    ci    = model.conf_int().loc[show]
    pvals = model.pvalues[show]
    colors = [GOLD if s == 'ACHP_Flag' else GREY for s in show]
    y_pos  = range(len(show))
    ax.barh(y_pos, coefs, color=colors, alpha=0.9, height=0.5)
    ax.errorbar(coefs, y_pos, xerr=[coefs-ci[0], ci[1]-coefs],
                fmt='none', color='#2c3e50', capsize=4, linewidth=1.5)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    for i, (coef, pv) in enumerate(zip(coefs, pvals)):
        stars = '***' if pv<0.001 else ('**' if pv<0.01 else ('*' if pv<0.05 else 'ns'))
        ax.text(coef+(max(abs(coefs))*0.05), i, stars, va='center', fontsize=10, fontweight='bold')
    ax.set_yticks(y_pos); ax.set_yticklabels([s.replace('_',' ') for s in show], fontsize=11)
    ax.set_xlabel('Coefficient ($)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
buf4 = io.BytesIO()
plt.savefig(buf4, format='png', dpi=150, bbox_inches='tight'); buf4.seek(0)
charts_f1['chart4_regression_coefs'] = buf4
plt.savefig(f'{OUT}chart4_regression_coefs.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('chart4_regression_coefs.png saved')


chart4_regression_coefs.png saved


## Cell 12 — Finding 1: Export to Excel

In [15]:
out_f1 = f'{OUT}finding1_viz_ml_output.xlsx'

def full_results_table(model, label):
    return pd.DataFrame({
        'Variable':    model.params.index,
        'Coefficient': model.params.values,
        'Std_Error':   model.bse.values,
        'p_value':     model.pvalues.values,
        'CI_2.5':      model.conf_int()[0].values,
        'CI_97.5':     model.conf_int()[1].values,
    }).round(5).assign(Significant_p05=lambda d: d['p_value']<0.05, Model=label)

model_fit = pd.DataFrame([{
    'Model':'Delta_Consolidated_Premium',
    'N_obs':int(model_premium.nobs), 'R_squared':round(model_premium.rsquared,4),
    'ACHP_Coef':round(achp_coef_p,4), 'ACHP_pvalue':round(achp_pval_p,6),
    'ACHP_CI_low':round(achp_ci_p[0],4), 'ACHP_CI_high':round(achp_ci_p[1],4),
    'ACHP_Significant':achp_pval_p<0.05,
},{
    'Model':'Delta_MOOP',
    'N_obs':int(model_moop.nobs), 'R_squared':round(model_moop.rsquared,4),
    'ACHP_Coef':round(achp_coef_m,4), 'ACHP_pvalue':round(achp_pval_m,6),
    'ACHP_CI_low':round(achp_ci_m[0],4), 'ACHP_CI_high':round(achp_ci_m[1],4),
    'ACHP_Significant':achp_pval_m<0.05,
}])

with pd.ExcelWriter(out_f1, engine='openpyxl') as writer:
    bar_df.to_excel(writer,                         sheet_name='chart1_cohort_bar_data',   index=False)
    model_fit.to_excel(writer,                      sheet_name='regression_model_summary', index=False)
    full_results_table(model_premium,'Delta_Premium').to_excel(writer, sheet_name='regression_premium_full', index=False)
    full_results_table(model_moop,   'Delta_MOOP').to_excel(writer,    sheet_name='regression_moop_full',    index=False)
    summary_p_df.to_excel(writer,                   sheet_name='regression_key_coefs',     index=False)
    wb  = writer.book
    cws = wb.create_sheet('charts')
    row_offset = 1
    for name, buf in charts_f1.items():
        buf.seek(0)
        img = XLImage(buf); img.width=750; img.height=450
        cws.add_image(img, f'A{row_offset}')
        row_offset += 30

print(f'Saved: finding1_viz_ml_output.xlsx  ({os.path.getsize(out_f1)//1024} KB)')


Saved: finding1_viz_ml_output.xlsx  (409 KB)


## Cell 13 — Finding 2: Drug Benefit Erosion

In [16]:
# ── BUG FIX: keep ACHP rows by treating null deductible delta as 0 ──
f2 = yoy.copy()
f2['Delta_PartD_Deductible'] = pd.to_numeric(f2['Delta_PartD_Deductible'], errors='coerce').fillna(0)
f2['Drug_Erosion_Flag'] = (f2['Delta_PartD_Deductible'] > 0).astype(int)
f2['Cohort_F2'] = f2['Cohort'].map({'ACHP-KP':'ACHP','ACHP-nonKP':'ACHP','National':'National'}).fillna('National')

w_col = 'Enrollment_2025'
GOLD = '#D4A017'; GREY = '#A8A8A8'

def f2_stats(df, cohort_col='Cohort_F2'):
    rows = []
    for cohort in sorted(df[cohort_col].unique()):
        sub = df[df[cohort_col]==cohort]
        w = sub[w_col] if w_col in sub.columns else pd.Series(1, index=sub.index)
        rows.append({
            'Cohort': cohort,
            'Rate_Flag_Any_Drug_Benefit_Erosion_pct': sub['Drug_Erosion_Flag'].mean()*100,
            'Weighted_Delta_Deductible': weighted_mean(sub['Delta_PartD_Deductible'], w),
            'N_Plans': len(sub),
        })
    return pd.DataFrame(rows)

df_f2 = f2_stats(f2)
df_f2_kp = f2_stats(f2[f2['Cohort'].isin(['ACHP-KP','ACHP-nonKP'])], cohort_col='Cohort')

print('Finding 2 Stats:'); print(df_f2.to_string(index=False))
print('\nWithin-ACHP KP split:'); print(df_f2_kp.to_string(index=False))

# ── Guard for missing cohort ──
if 'ACHP' not in df_f2['Cohort'].values:
    print('WARNING: No ACHP rows in df_f2 after fix — check yoy merge.')
else:
    df_plot = df_f2[df_f2['Cohort'].isin(['ACHP','National'])].copy()
    f2_colors = {'ACHP': GREY, 'National': GOLD}

    # Visual 1 — Erosion rate
    plt.figure(figsize=(8,6))
    ax1 = sns.barplot(x='Cohort', y='Rate_Flag_Any_Drug_Benefit_Erosion_pct',
                      data=df_plot,
                      palette=f2_colors,
                      order=['ACHP','National'])
    plt.title('% of Plans with Drug Benefit Erosion\n(any Part D deductible increase 2025->2026)',
              fontsize=15, fontweight='normal', pad=15)
    plt.ylabel(''); plt.xlabel(''); plt.ylim(0,100)
    for p in ax1.patches:
        ax1.annotate(f'{p.get_height():.1f}%',
                     (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=14, fontweight='bold',
                     xytext=(0,5), textcoords='offset points')
    ax1.yaxis.set_visible(False); ax1.grid(False)
    sns.despine(left=True); plt.tight_layout()
    plt.savefig(f'{OUT}finding2_benefit_erosion.png', dpi=300)
    plt.show(); plt.close()
    print('finding2_benefit_erosion.png saved')

    # Visual 2 — Weighted deductible increase
    plt.figure(figsize=(8,6))
    ax2 = sns.barplot(x='Cohort', y='Weighted_Delta_Deductible',
                      data=df_plot,
                      palette=f2_colors,
                      order=['ACHP','National'])
    plt.title('Enrollment-Weighted Part D Deductible Increase\n2025 -> 2026',
              fontsize=15, fontweight='normal', pad=20)
    plt.ylabel(''); plt.xlabel('')
    for p in ax2.patches:
        ax2.annotate(f'${p.get_height():.2f}',
                     (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=14, fontweight='bold',
                     xytext=(0,5), textcoords='offset points')
    ax2.yaxis.set_visible(False); ax2.grid(False)
    sns.despine(left=True); plt.tight_layout()
    plt.savefig(f'{OUT}finding2_deductible_increase.png', dpi=300)
    plt.show(); plt.close()
    print('finding2_deductible_increase.png saved')

# Visual 3 — Top carriers: deductible increase vs erosion rate
W_COL = 'Enrollment_2025'
ORG_COL = 'Organization Marketing Name_2025'

f2['Delta_PartD_Deductible'] = pd.to_numeric(f2['Delta_PartD_Deductible'], errors='coerce').fillna(0)
f2['Drug_Erosion_Flag'] = (f2['Delta_PartD_Deductible'] > 0).astype(int)

def weighted_mean(values, weights):
    w = weights.fillna(0)
    return (values * w).sum() / w.sum() if w.sum() > 0 else values.mean()

# ACHP benchmark
achp_sub = f2[f2['Cohort_F2'] == 'ACHP']
achp_ero_b = achp_sub['Drug_Erosion_Flag'].mean() * 100
achp_ded_b = weighted_mean(achp_sub['Delta_PartD_Deductible'], achp_sub[W_COL])

national = f2[f2['Cohort_F2'] == 'National'].copy()
org_upper = national[ORG_COL].fillna('').str.upper()

brand_patterns = {
    'UnitedHealthcare':      'UNITEDHEALTHCARE',
    'Humana':                'HUMANA',
    'Aetna Medicare':        'AETNA',
    'Anthem':                'ANTHEM|WELLPOINT',
    'Centene':               'CENTENE|WELLCARE|SIMPLY HEALTHCARE',
    'Blue Cross Blue Shield': 'BLUE CROSS|BLUE SHIELD|BLUECROSS|BLUESHIELD',
    'Cigna':                 'CIGNA|HEALTHSPRING',
    'Molina Healthcare':     'MOLINA',
    'Healthfirst':           'HEALTHFIRST',
    'SCAN Health Plan':      'SCAN',
}

rows = []
for brand, pattern in brand_patterns.items():
    mask = org_upper.str.contains(pattern, regex=True)
    sub = national[mask]
    if len(sub) == 0:
        print(f'WARNING: No rows matched for: {brand}')
        continue
    w = sub[W_COL].fillna(0)
    rows.append({
        'Brand':               brand,
        'deductible_increase': weighted_mean(sub['Delta_PartD_Deductible'], w),
        'erosion_pct':         sub['Drug_Erosion_Flag'].mean() * 100,
        'is_kaiser':           False,
    })

carrier_stats = pd.DataFrame(rows).sort_values('deductible_increase')

fig, ax = plt.subplots(figsize=(12, 8))
for _, row in carrier_stats.iterrows():
    color = GOLD
    ax.scatter(row['deductible_increase'], row['erosion_pct'], color=color, s=120, zorder=5)
    ax.annotate(row['Brand'], (row['deductible_increase'], row['erosion_pct']),
                textcoords='offset points', xytext=(8, 0),
                fontsize=9, color=color, fontweight='bold')

ax.axhline(achp_ero_b, color='#555555', linestyle='--', linewidth=1.2, alpha=0.7)
ax.axvline(achp_ded_b, color='#555555', linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(achp_ded_b + 2, achp_ero_b - 3,
        f'ACHP benchmark',
        fontsize=8, color='#555555', style='italic')
ax.set_xlabel('Enrollment-Weighted Avg Deductible Increase ($)', fontsize=12)
ax.set_ylabel('% of Plans with Erosion', fontsize=12)
ax.set_title('Top 10 National Carriers: Deductible Increase & Erosion Rate', fontsize=13, pad=20)
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('finding2_scatter.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved → finding2_scatter.png')

# ── Panel: dual horizontal bar chart + table ─────────────────────────────────
df_panel = carrier_stats[~carrier_stats['is_kaiser']].sort_values('deductible_increase', ascending=True)

orgs           = df_panel['Brand'].tolist()
short_orgs     = orgs
deductible_inc = df_panel['deductible_increase'].tolist()
erosion_vals   = df_panel['erosion_pct'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

y_pos = np.arange(len(orgs))
bar_h = 0.35

bar_colors_ded = [GOLD if d > achp_ded_b else GREY for d in deductible_inc]
bar_colors_ero = [GOLD if e > achp_ero_b else GREY for e in erosion_vals]

ax = axes[0]
b1 = ax.barh(y_pos + bar_h/2, deductible_inc, bar_h,
             color=bar_colors_ded, alpha=0.9, label='Avg Deductible Increase ($)', edgecolor='white')
b2 = ax.barh(y_pos - bar_h/2, erosion_vals,   bar_h,
             color=bar_colors_ero, alpha=0.55, label='% Plans with Erosion', edgecolor='white')

for bar, val in zip(b1, deductible_inc):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f'${val:.0f}', va='center', fontsize=8.5, fontweight='bold')
for bar, val in zip(b2, erosion_vals):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=8.5)

ax.axvline(achp_ded_b, color='#333', linestyle='--', linewidth=1.1, alpha=0.7,
           label=f'ACHP deductible benchmark (${achp_ded_b:.0f})')
ax.axvline(achp_ero_b, color='#888', linestyle=':', linewidth=1.1, alpha=0.7,
           label=f'ACHP erosion benchmark ({achp_ero_b:.0f}%)')

ax.set_yticks(y_pos)
ax.set_yticklabels(short_orgs, fontsize=9)
ax.set_xlabel('Value ($ or %)', fontsize=10)
ax.legend(fontsize=8, loc='lower right', frameon=True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.invert_yaxis()

# ── Panel B: summary table ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.axis('off')

table_data = [['Carrier', 'Ded. Increase', 'Erosion %']]
for org, ded, ero in zip(short_orgs, deductible_inc, erosion_vals):
    table_data.append([org, f'${ded:.0f}', f'{ero:.0f}%'])
table_data.append(['— ACHP Benchmark —', f'${achp_ded_b:.0f}', f'{achp_ero_b:.0f}%'])

col_widths = [0.58, 0.22, 0.20]
tbl = ax2.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    colWidths=col_widths,
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
tbl.scale(1, 1.38)

for j in range(3):
    tbl[(0, j)].set_facecolor('#1E2761')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

for j in range(3):
    tbl[(len(orgs)+1, j)].set_facecolor('#F5F5DC')
    tbl[(len(orgs)+1, j)].set_text_props(style='italic', fontsize=8)

for i, (ded, ero) in enumerate(zip(deductible_inc, erosion_vals)):
    if ded > achp_ded_b or ero > achp_ero_b:
        for j in range(3):
            tbl[(i+1, j)].set_facecolor('#FFF3CD')

ax2.set_title('Reference Table', fontsize=10, fontweight='bold', pad=8)

plt.suptitle('Top 10 National Carriers: Drug Benefit Erosion vs ACHP Benchmark',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}finding2_panel.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('finding2_panel.png saved')

Finding 2 Stats:
  Cohort  Rate_Flag_Any_Drug_Benefit_Erosion_pct  Weighted_Delta_Deductible  N_Plans
    ACHP                               41.340442                  25.622134     6565
National                               67.495364                 114.097135   102450

Within-ACHP KP split:
    Cohort  Rate_Flag_Any_Drug_Benefit_Erosion_pct  Weighted_Delta_Deductible  N_Plans
   ACHP-KP                               13.086420                 -26.687464      405
ACHP-nonKP                               43.198052                  77.990956     6160
finding2_benefit_erosion.png saved
finding2_deductible_increase.png saved
Saved → finding2_scatter.png
finding2_panel.png saved


## Cell 14 — Finding 3: HMO Competitive Cluster Analysis Setup
Filters to HMO/HMO-POS non-SNP plans. KP flagged for overlay. All clusters shown.


In [17]:
CLUSTER_FEATURES = [
    'Monthly Consolidated Premium (Part C + D)_2026',
    'In-Network Maximum Out-of-Pocket (MOOP) Amount_2026',
    'Overall Star Rating_2026',
    'Enrollment_2026',
    'Annual Part D Deductible Amount_2026',
]

pt_col  = 'Plan Type_2026' if 'Plan Type_2026' in yoy.columns else None
snp_col = 'SNP Type_2026'  if 'SNP Type_2026'  in yoy.columns else None

hmo_mask = pd.Series(True, index=yoy.index)
if pt_col:
    hmo_mask = yoy[pt_col].astype(str).str.upper().str.contains('HMO')
if snp_col:
    hmo_mask = hmo_mask & (yoy[snp_col].astype(str).str.upper().str.contains('NOT APPLICABLE|NAN|NONE'))

enrl_mask = yoy['Enrollment_2026'].fillna(0) > 0
hmo_clean = yoy[hmo_mask & enrl_mask].copy()

avail_feats = [f for f in CLUSTER_FEATURES if f in hmo_clean.columns]
hmo_clean   = hmo_clean.dropna(subset=avail_feats)

# Ensure ACHP_All and KP_Flag columns present
if 'ACHP_All' not in hmo_clean.columns:
    achp_col_hmo = 'ACHP?_2026' if 'ACHP?_2026' in hmo_clean.columns else 'ACHP?_2025'
    hmo_clean['ACHP_All'] = (hmo_clean[achp_col_hmo] == 1).astype(int)
if 'KP_Flag' not in hmo_clean.columns:
    hmo_clean['KP_Flag'] = 0

print(f'HMO/HMO-POS non-SNP rows after feature dropna: {len(hmo_clean):,}')
print(f'  ACHP-KP:    {(hmo_clean["Cohort"]=="ACHP-KP").sum():,}')
print(f'  ACHP-nonKP: {(hmo_clean["Cohort"]=="ACHP-nonKP").sum():,}')
print(f'  National:   {(hmo_clean["Cohort"]=="National").sum():,}')
print('\nCluster features:')
for f in avail_feats: print(f'  {f}')


HMO/HMO-POS non-SNP rows after feature dropna: 18,993
  ACHP-KP:    262
  ACHP-nonKP: 1,175
  National:   17,556

Cluster features:
  Monthly Consolidated Premium (Part C + D)_2026
  In-Network Maximum Out-of-Pocket (MOOP) Amount_2026
  Overall Star Rating_2026
  Enrollment_2026
  Annual Part D Deductible Amount_2026


## Cell 15 — Finding 3: Elbow / Silhouette

In [18]:
# Force all cluster feature columns to numeric before scaling
# (removes strings like 'Not Enough Data Available')
for feat in avail_feats:
    hmo_clean[feat] = pd.to_numeric(hmo_clean[feat], errors='coerce')

# Re-drop rows that became NaN after numeric conversion
hmo_clean = hmo_clean.dropna(subset=avail_feats).copy()
print(f'Rows after numeric cleaning: {len(hmo_clean):,}')

X          = hmo_clean[avail_feats].values
pipe_scale = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())])
X_scaled   = pipe_scale.fit_transform(X)

K_RANGE = range(2,9)
inertias, sil_scores, db_scores = [], [], []
for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl))
    db_scores.append(davies_bouldin_score(X_scaled, lbl))

fig, axes = plt.subplots(1,3, figsize=(15,4))
axes[0].plot(list(K_RANGE), inertias,  'bo-'); axes[0].set_title('Elbow (Inertia)');            axes[0].set_xlabel('k')
axes[1].plot(list(K_RANGE), sil_scores,'go-'); axes[1].set_title('Silhouette Score');            axes[1].set_xlabel('k')
axes[2].plot(list(K_RANGE), db_scores, 'ro-'); axes[2].set_title('Davies-Bouldin (lower=better)'); axes[2].set_xlabel('k')
plt.suptitle('Cluster Validation — HMO/HMO-POS SNP Excluded', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}finding3_elbow.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

best_k = list(K_RANGE)[np.argmax(sil_scores)]
print(f'Best k by silhouette: {best_k}  (score={max(sil_scores):.4f})')
print('Override in Cell 16 if desired: best_k = 5')

Rows after numeric cleaning: 18,725
Best k by silhouette: 2  (score=0.3219)
Override in Cell 16 if desired: best_k = 5


## Cell 16 — Finding 3: Fit Clusters + KP Overlay Visualization

In [19]:
# Point: which cluster(s) contain ACHP/KP plans — gold overlay line for ACHP%, grey clusters
GOLD = '#D4A017'; GREY_CLUSTER = '#A8A8A8'; KP_COLOR = '#D4A017'
GREY_SHADES = ['#A8A8A8','#B8B8B8','#C0C0C0','#C8C8C8','#D0D0D0',
               '#D8D8D8','#BEBEBE','#969696','#888888','#909090']

# best_k = 5  # uncomment to override

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
hmo_clean = hmo_clean.copy()
hmo_clean['Cluster'] = km_final.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_scaled)
hmo_clean['PC1'], hmo_clean['PC2'] = pcs[:,0], pcs[:,1]

cluster_sum = hmo_clean.groupby('Cluster').apply(lambda x: pd.Series({
    'N_Plans':         len(x),
    'ACHP_Pct':        round((x['ACHP_All']==1).mean()*100,1),
    'KP_Pct':          round((x['KP_Flag']==1).mean()*100,1),
    'Mean_Premium':    round(x['Monthly Consolidated Premium (Part C + D)_2026'].mean(),2),
    'Mean_MOOP':       round(x['In-Network Maximum Out-of-Pocket (MOOP) Amount_2026'].mean(),0),
    'Mean_Stars':      round(x['Overall Star Rating_2026'].mean(),2),
    'Mean_Deductible': round(x['Annual Part D Deductible Amount_2026'].mean(),2),
})).reset_index()

print(f'=== CLUSTER PROFILES (all {best_k} clusters) ===')
print(f'Features: {", ".join([f.split("_2026")[0].split(" (")[0] for f in avail_feats])}')
print(cluster_sum.to_string(index=False))

fig, axes = plt.subplots(1,2, figsize=(18,7))

# Panel A: PCA scatter — all clusters grey, KP stars gold
ax = axes[0]
for cl in sorted(hmo_clean['Cluster'].unique()):
    sub    = hmo_clean[hmo_clean['Cluster']==cl]
    non_kp = sub[sub['KP_Flag']==0]
    shade  = GREY_SHADES[cl % len(GREY_SHADES)]
    ax.scatter(non_kp['PC1'], non_kp['PC2'], c=[shade], s=18, alpha=0.5, label=f'Cluster {cl}')
    kp = sub[sub['KP_Flag']==1]
    if len(kp) > 0:
        ax.scatter(kp['PC1'], kp['PC2'], c=[GOLD], s=80, alpha=0.95, marker='*', zorder=6)
ax.scatter([],[],c=GOLD, s=80, marker='*', label='Kaiser Permanente')
ax.set_xlabel('PC1',fontsize=9); ax.set_ylabel('PC2',fontsize=9)
ax.set_title(f'PCA — {best_k} Clusters | star=KP | SNPs excluded', fontsize=10, fontweight='bold')
ax.legend(fontsize=7, loc='lower right'); ax.tick_params(labelsize=8)

# Panel B: Profile bars — grey for metric bars, gold line for ACHP%
ax2  = axes[1]
x    = np.arange(best_k); w = 0.22
bar_grey1 = '#B0B0B0'; bar_grey2 = '#C8C8C8'; bar_grey3 = '#D8D8D8'
ax2.bar(x-w, cluster_sum['Mean_Stars'],       w, color=bar_grey1, alpha=0.9,  label='Avg Stars')
ax2.bar(x,   cluster_sum['Mean_Premium']/100, w, color=bar_grey2, alpha=0.85, label='Avg Premium (÷100)')
ax2.bar(x+w, cluster_sum['Mean_MOOP']/1000,   w, color=bar_grey3, alpha=0.85, label='Avg MOOP (÷1000)')

ax2b = ax2.twinx()
ax2b.plot(x, cluster_sum['ACHP_Pct'], 'o--', color=GOLD,   linewidth=2.5, markersize=9, label='ACHP %')
ax2b.plot(x, cluster_sum['KP_Pct'],   's--', color='#888', linewidth=1.5, markersize=7, label='KP %')
ax2b.set_ylabel('% of Cluster', fontsize=9)
ax2b.set_ylim(0, max(cluster_sum['ACHP_Pct'].max(), cluster_sum['KP_Pct'].max())*2.2)

ax2.set_xticks(x); ax2.set_xticklabels([f'Cluster {i}' for i in range(best_k)], fontsize=8)
ax2.set_title(f'Cluster Profiles: Stars / Premium / MOOP\n'
              f'Features: {", ".join([f.split("_2026")[0].split(" (")[0] for f in avail_feats])}',
              fontsize=10, fontweight='bold')
lines1,lbl1 = ax2.get_legend_handles_labels()
lines2,lbl2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, lbl1+lbl2, fontsize=7.5, loc='upper right')

fig.suptitle(f'Finding 3 — HMO/HMO-POS Cluster Analysis (2026) | k={best_k} | KP isolated',
             fontsize=11, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(f'{OUT}finding3_viz_snp_excluded_clusters.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('finding3_viz_snp_excluded_clusters.png saved')

=== CLUSTER PROFILES (all 2 clusters) ===
Features: Monthly Consolidated Premium, In-Network Maximum Out-of-Pocket, Overall Star Rating, Enrollment, Annual Part D Deductible Amount
 Cluster  N_Plans  ACHP_Pct  KP_Pct  Mean_Premium  Mean_MOOP  Mean_Stars  Mean_Deductible
       0   6116.0      19.7     4.4         33.18     4159.0        4.29           166.74
       1  12609.0       1.8     0.0          7.01     6294.0        3.64           505.47
finding3_viz_snp_excluded_clusters.png saved


## Cell 17 — Finding 3: Churn / Plan Exit Logistic Regression

In [20]:
ids_26 = set((df26['Contract ID'].astype(str)+'_'+df26['Plan ID'].astype(str)).unique())

df25_churn = df25.copy()
if 'ContractPlanID' not in df25_churn.columns:
    df25_churn['ContractPlanID'] = (df25_churn['Contract ID'].astype(str)+'_'+
                                    df25_churn['Plan ID'].astype(str))

df25_churn['Churned']      = (~df25_churn['ContractPlanID'].isin(ids_26)).astype(int)
df25_churn['Cohort_Label'] = df25_churn['ACHP?'].map({1:'ACHP',0:'National/Regional'}).fillna('National/Regional')

print(f'Plans in 2025: {len(df25_churn):,}')
print(f'Churned:       {df25_churn["Churned"].sum():,} ({df25_churn["Churned"].mean()*100:.1f}%)')

pt_col_25 = 'Plan Type' if 'Plan Type' in df25_churn.columns else None
if pt_col_25:
    churn_by_cohort = (df25_churn
        .groupby(['Cohort_Label',pt_col_25])
        .agg(Plan_Count=('Churned','count'), Churned_Count=('Churned','sum'), Churn_Rate=('Churned','mean'))
        .reset_index())
    churn_by_cohort['Churn_Rate_Pct'] = (churn_by_cohort['Churn_Rate']*100).round(1)
    main_types = ['HMO','HMO-POS','PPO','Regional PPO','HMO D-SNP','HMO-POS D-SNP']
    churn_plot = churn_by_cohort[churn_by_cohort[pt_col_25].isin(main_types)].copy()
else:
    churn_plot = pd.DataFrame()

CHURN_FEATS    = ['Monthly Consolidated Premium (Part C + D)',
                  'In-Network Maximum Out-of-Pocket (MOOP) Amount',
                  'Overall Star Rating','Enrollment','ACHP?']
avail_lr_feats = [f for f in CHURN_FEATS if f in df25_churn.columns]
df_lr          = df25_churn[avail_lr_feats+['Churned']].copy().dropna(subset=['Churned'])
for c in avail_lr_feats:
    df_lr[c] = pd.to_numeric(df_lr[c], errors='coerce')

y_lr = df_lr['Churned'].values
X_lr = df_lr[avail_lr_feats].values

pipe_lr = Pipeline([
    ('imp',   SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('lr',    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=0.5))
])
auc_scores = cross_val_score(pipe_lr, X_lr, y_lr, scoring='roc_auc',
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))
pipe_lr.fit(X_lr, y_lr)
lr_model = pipe_lr.named_steps['lr']
coef_df  = pd.DataFrame({'Feature':avail_lr_feats, 'Coefficient':lr_model.coef_[0],
                          'Odds_Ratio':np.exp(lr_model.coef_[0])
                         }).sort_values('Coefficient', ascending=False).reset_index(drop=True)

print(f'\n5-fold CV AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}')
print(coef_df.to_string(index=False))


Plans in 2025: 142,653
Churned:       119,175 (83.5%)

5-fold CV AUC: 0.6416 +/- 0.0031
                                       Feature  Coefficient  Odds_Ratio
     Monthly Consolidated Premium (Part C + D)     0.524714    1.689975
                                         ACHP?     0.366000    1.441955
                                    Enrollment     0.009838    1.009886
                           Overall Star Rating    -0.111978    0.894064
In-Network Maximum Out-of-Pocket (MOOP) Amount    -0.259306    0.771587


## Cell 18 — Finding 3: Churn Visualization

In [21]:
# ============================================================
# Cell 18 — Finding 3: Full Analysis (Regional cohort, 3×3
# heatmap, county competitive analysis, corrected clustering,
# corrected plan-level churn LR, visualizations, Excel export)
# ============================================================

import json, urllib.request, warnings
warnings.filterwarnings("ignore")
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.collections import PatchCollection
from matplotlib.colors import ListedColormap
from sklearn.metrics import silhouette_score
from pathlib import Path as _Path

# ── Color palette ────────────────────────────────────────────
GOLD      = "#D4A017"
GREY_MID  = "#A8A8A8"
GREY_DARK = "#888888"
GREY_BG   = "#F5F5F5"
GREY_LINE = "#2c3e50"
FONT_T, FONT_L, FONT_K = 16, 12, 10

COHORT_COLORS = {
    "ACHP-KP":    GOLD, "ACHP-nonKP": GOLD,
    "ACHP":       GOLD, "Regional":   GREY_MID,
    "National":   GREY_MID,
}
COHORT_LABELS = {
    "ACHP-KP":    "Kaiser-only ACHP",  "ACHP-nonKP": "Non-Kaiser ACHP",
    "ACHP":       "All ACHP",          "Regional":   "Regional competitors",
    "National":   "National competitors",
}
PLAN_LABELS = {
    "HMO": "Local HMO", "HMO-POS": "HMO-POS",
    "PPO": "PPO",        "Regional PPO": "Regional PPO",
}
GROUP_LABELS = {
    "All ACHP":       "All ACHP",
    "ACHP excl. KP":  "Non-Kaiser ACHP",
    "Kaiser only":    "Kaiser-only ACHP",
    "National":       "National competitors",
}
FEATURE_LABELS = {
    "Enrollment":                                      "Enrollment",
    "Overall Star Rating":                             "Star rating",
    "Monthly Consolidated Premium (Part C + D)":      "Monthly premium",
    "In-Network Maximum Out-of-Pocket (MOOP) Amount": "MOOP",
    "ACHP?":     "ACHP member plan",
    "KP_Flag":   "Kaiser flag",
    "PT_Enc":    "Plan type",
    "Benefit_OTC":     "OTC benefit",   "Benefit_Dental":  "Dental benefit",
    "Benefit_Hearing": "Hearing benefit","Benefit_Food":    "Food benefit",
}

# ── Column aliases ───────────────────────────────────────────
W      = "Enrollment_2025"
prem26 = "Monthly Consolidated Premium (Part C + D)_2026"
moop26 = "In-Network Maximum Out-of-Pocket (MOOP) Amount_2026"
star26 = "Overall Star Rating_2026"
HERE   = _Path(".")
OUT3   = OUT

# Normalise delta column names
if "Delta_Premium" not in yoy.columns and "Delta_Consolidated_Premium" in yoy.columns:
    yoy["Delta_Premium"] = yoy["Delta_Consolidated_Premium"]
if "Delta_Deductible" not in yoy.columns and "Delta_PartD_Deductible" in yoy.columns:
    yoy["Delta_Deductible"] = yoy["Delta_PartD_Deductible"]
if "Combined_Cost_Increase" not in yoy.columns:
    yoy["Combined_Cost_Increase"] = yoy["Delta_Premium"].fillna(0) + yoy["Delta_MOOP"].fillna(0)

# Force star26 to numeric in yoy to prevent TypeError downstream
if star26 in yoy.columns:
    yoy[star26] = pd.to_numeric(yoy[star26], errors="coerce")

# ── Enrollment-weighted mean helper ──────────────────────────
def wm(series, weights):
    s = pd.to_numeric(series, errors="coerce")
    mask = s.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0: return np.nan
    return float(np.average(s[mask], weights=weights[mask]))

# ============================================================
# STEP 1 — Add Regional cohort
# ============================================================
org_col   = ("Organization Type_2026" if "Organization Type_2026" in yoy.columns
             else "Organization Type_2025" if "Organization Type_2025" in yoy.columns
             else None)
achp_col  = "ACHP?_2026" if "ACHP?_2026" in yoy.columns else "ACHP?_2025"
achp_mask = yoy[achp_col] == 1
reg_mask  = (pd.Series(False, index=yoy.index) if org_col is None
             else yoy[org_col].fillna("").str.contains("Regional", case=False) & ~achp_mask)

yoy["Cohort"] = "National"
yoy.loc[reg_mask,                              "Cohort"] = "Regional"
yoy.loc[achp_mask & (yoy["KP_Flag"] == 0),    "Cohort"] = "ACHP-nonKP"
yoy.loc[achp_mask & (yoy["KP_Flag"] == 1),    "Cohort"] = "ACHP-KP"
yoy["ACHP_All"] = achp_mask.astype(int)

print("Cohort distribution:")
print(yoy["Cohort"].value_counts().to_string())

# ============================================================
# STEP 2 — Plan Type × Cohort cross-tab (3×3 + detailed)
# ============================================================
pt_col = "Plan Type_2026" if "Plan Type_2026" in yoy.columns else "Plan Type_2025"

def simplify_pt(s):
    s = str(s).strip()
    if "HMO-POS" in s and "SNP" not in s: return "HMO-POS"
    if s == "HMO":                         return "HMO"
    if "Regional PPO" in s:                return "Regional PPO"
    if s == "PPO":                         return "PPO"
    return None

yoy["PlanType_Simple"] = yoy[pt_col].apply(simplify_pt)
yoy["Cohort3"] = yoy["Cohort"].replace({"ACHP-KP": "ACHP", "ACHP-nonKP": "ACHP"})

CROSSTAB_COHORTS = ["ACHP-KP", "ACHP-nonKP", "Regional", "National"]
CROSSTAB_TYPES   = ["HMO", "HMO-POS", "PPO", "Regional PPO"]
HEATMAP_TYPES    = ["HMO", "HMO-POS", "Regional PPO"]
HEATMAP_COHORTS  = ["ACHP", "Regional", "National"]

ct_rows = []
for pt in CROSSTAB_TYPES:
    for cohort in CROSSTAB_COHORTS:
        sub = yoy[(yoy["PlanType_Simple"] == pt) & (yoy["Cohort"] == cohort)]
        w   = sub[W].fillna(0)
        if len(sub) == 0:
            ct_rows.append({"Plan_Type": pt, "Cohort": cohort, "Plan_Count": 0,
                            "Total_Enrollment": 0, "Wtd_Delta_Premium": np.nan,
                            "Wtd_Delta_MOOP": np.nan, "Wtd_Delta_Deductible": np.nan,
                            "Combined_Cost_Increase": np.nan, "Pct_Plans_Cost_Shifted": np.nan})
            continue
        dp = wm(sub["Delta_Premium"],    w)
        dm = wm(sub["Delta_MOOP"],       w)
        dd = wm(sub["Delta_Deductible"], w)
        ct_rows.append({
            "Plan_Type": pt, "Cohort": cohort, "Plan_Count": len(sub),
            "Total_Enrollment": int(w.sum()),
            "Wtd_Delta_Premium":    round(dp, 2) if dp is not None else np.nan,
            "Wtd_Delta_MOOP":       round(dm, 2) if dm is not None else np.nan,
            "Wtd_Delta_Deductible": round(dd, 2) if dd is not None else np.nan,
            "Combined_Cost_Increase": round((dp or 0) + (dm or 0), 2),
            "Pct_Plans_Cost_Shifted": round((sub["Delta_Premium"].fillna(0) > 0).mean() * 100, 1),
        })
crosstab_df = pd.DataFrame(ct_rows)

m_rows = []
for pt in HEATMAP_TYPES:
    for cohort in HEATMAP_COHORTS:
        sub = yoy[(yoy["PlanType_Simple"] == pt) & (yoy["Cohort3"] == cohort)]
        w   = sub[W].fillna(0)
        if len(sub) == 0:
            m_rows.append({"Plan_Type": pt, "Cohort": cohort, "Plan_Count": 0,
                           "Total_Enrollment": 0, "Wtd_Delta_Premium": np.nan,
                           "Wtd_Delta_MOOP": np.nan, "Combined_Cost_Increase": np.nan})
            continue
        dp = wm(sub["Delta_Premium"], w); dm = wm(sub["Delta_MOOP"], w)
        m_rows.append({"Plan_Type": pt, "Cohort": cohort, "Plan_Count": len(sub),
                       "Total_Enrollment": int(w.sum()),
                       "Wtd_Delta_Premium": round(dp, 2), "Wtd_Delta_MOOP": round(dm, 2),
                       "Combined_Cost_Increase": round((dp or 0) + (dm or 0), 2)})
matrix_3x3_df = pd.DataFrame(m_rows)

valid_m = matrix_3x3_df.dropna(subset=["Combined_Cost_Increase"])
worst   = valid_m.loc[valid_m["Combined_Cost_Increase"].idxmax()] if len(valid_m) else None
if worst is not None:
    print(f"\nWorst cell: {worst['Plan_Type']} × {worst['Cohort']} = ${worst['Combined_Cost_Increase']:,.0f}")
print(crosstab_df[["Plan_Type","Cohort","Plan_Count","Wtd_Delta_Premium",
                   "Wtd_Delta_MOOP","Combined_Cost_Increase"]].to_string(index=False))

# ============================================================
# STEP 3 — Kaiser Isolation
# ============================================================
kaiser_rows = []
for label, mask in [
    ("All ACHP",       achp_mask),
    ("ACHP excl. KP",  achp_mask & (yoy["KP_Flag"] == 0)),
    ("Kaiser only",    achp_mask & (yoy["KP_Flag"] == 1)),
    ("National",       yoy["Cohort"] == "National"),
]:
    sub = yoy[mask]; w = sub[W].fillna(0)
    kaiser_rows.append({
        "Group":                    label,
        "Plan_Count":               len(sub),
        "Total_Enrollment":         int(w.sum()),
        "Wtd_Stars_2026":           round(wm(sub[star26], w), 3) if star26 in sub.columns else np.nan,
        "Wtd_Delta_Premium":        round(wm(sub["Delta_Premium"],    w), 2),
        "Wtd_Delta_MOOP":           round(wm(sub["Delta_MOOP"],       w), 2),
        "Wtd_Delta_Deductible":     round(wm(sub["Delta_Deductible"], w), 2),
        "Pct_Plans_Raised_Premium": round((sub["Delta_Premium"].fillna(0) > 0).mean() * 100, 1),
        "Pct_Plans_Raised_MOOP":    round((sub["Delta_MOOP"].fillna(0) > 0).mean() * 100, 1),
    })
kaiser_df = pd.DataFrame(kaiser_rows)
print("\nKaiser isolation:")
print(kaiser_df[["Group","Wtd_Delta_Premium","Wtd_Delta_MOOP","Pct_Plans_Raised_MOOP"]].to_string(index=False))

# ============================================================
# STEP 4 — County-Level Competitive Analysis
# ============================================================
def county_achp_rollup(group):
    w = group[W].fillna(0)
    return pd.Series({
        "ACHP_Min_Premium":  group[prem26].min(),
        "ACHP_Min_MOOP":     group[moop26].min(),
        "ACHP_Avg_Stars":    pd.to_numeric(group[star26], errors="coerce").mean(),
        "ACHP_Wtd_Stars":    wm(group[star26], w),
        "ACHP_Plan_Count":   len(group),
        "ACHP_Enrollment":   int(w.sum()),
    })

achp_county = (
    yoy[achp_mask & yoy[prem26].notna()]
    .groupby(["State Abbreviation", "County Name"], group_keys=False)
    .apply(county_achp_rollup).reset_index()
)

natl_mask     = yoy["Cohort"] == "National"
hmo_peer_mask = natl_mask & yoy["PlanType_Simple"].isin(["HMO", "HMO-POS"])
org_col_26    = "Organization Marketing Name_2026"

def direction_label(d):
    if pd.isna(d): return "No data"
    return "Worsened" if d > 0 else ("Improved" if d < 0 else "Flat")

def county_competitor_rollup(group, cset):
    if org_col_26 in group.columns and group[org_col_26].notna().any():
        top = group.groupby(org_col_26)[W].sum().idxmax() if W in group.columns else group[org_col_26].mode().iloc[0]
        cg  = group[group[org_col_26] == top].copy()
    else:
        top = "Unknown"; cg = group.copy()
    cp = cg["PlanType_Simple"].dropna().mode()
    dp = wm(cg["Delta_Premium"], cg[W].fillna(0))
    dm = wm(cg["Delta_MOOP"],    cg[W].fillna(0))
    return pd.Series({
        "Competitor_Set": cset, "Top_Competitor": top,
        "Top_Competitor_Plan_Type":  cp.iloc[0] if not cp.empty else "Unknown",
        "Competitor_Min_Premium":    cg[prem26].min(),
        "Competitor_Min_MOOP":       cg[moop26].min(),
        "Competitor_Avg_Stars":      pd.to_numeric(cg[star26], errors="coerce").mean(),
        "Competitor_Wtd_Stars":      wm(cg[star26], cg[W].fillna(0)),
        "Competitor_Plan_Count":     len(cg),
        "Competitor_Enrollment":     int(cg[W].fillna(0).sum()),
        "Competitor_Delta_Premium":  dp,
        "Competitor_Delta_MOOP":     dm,
        "Competitor_Premium_Direction": direction_label(dp),
        "Competitor_MOOP_Direction":    direction_label(dm),
    })

hmo_peer_county = (yoy[hmo_peer_mask & yoy[prem26].notna()]
    .groupby(["State Abbreviation", "County Name"], group_keys=False)
    .apply(lambda g: county_competitor_rollup(g, "National HMO/HMO-POS")).reset_index())
fallback_county = (yoy[natl_mask & yoy[prem26].notna()]
    .groupby(["State Abbreviation", "County Name"], group_keys=False)
    .apply(lambda g: county_competitor_rollup(g, "National fallback")).reset_index())

county_df = (achp_county
    .merge(hmo_peer_county, on=["State Abbreviation","County Name"], how="left")
    .merge(fallback_county,  on=["State Abbreviation","County Name"], how="left", suffixes=("","_fallback")))

fill_cols = ["Competitor_Set","Top_Competitor","Top_Competitor_Plan_Type",
             "Competitor_Min_Premium","Competitor_Min_MOOP","Competitor_Avg_Stars",
             "Competitor_Wtd_Stars","Competitor_Plan_Count","Competitor_Enrollment",
             "Competitor_Delta_Premium","Competitor_Delta_MOOP",
             "Competitor_Premium_Direction","Competitor_MOOP_Direction"]
for col in fill_cols:
    county_df[col] = county_df[col].combine_first(county_df.get(f"{col}_fallback"))
county_df = county_df.drop(columns=[c for c in county_df.columns if c.endswith("_fallback")])
county_df = county_df.dropna(subset=["Top_Competitor"]).reset_index(drop=True)

county_df["Premium_Advantage"]  = county_df["Competitor_Min_Premium"] - county_df["ACHP_Min_Premium"]
county_df["MOOP_Advantage"]     = county_df["Competitor_Min_MOOP"]    - county_df["ACHP_Min_MOOP"]
county_df["Star_Advantage"]     = county_df["ACHP_Avg_Stars"]         - county_df["Competitor_Avg_Stars"]
county_df["Star_Advantage_Wtd"] = county_df["ACHP_Wtd_Stars"]         - county_df["Competitor_Wtd_Stars"]
county_df["Competitive_Score"]  = (
    county_df["Premium_Advantage"].fillna(0) +
    county_df["MOOP_Advantage"].fillna(0) * 0.1 +
    county_df["Star_Advantage_Wtd"].fillna(0) * 100
)
print(f"\nCounties analysed: {len(county_df):,}")
print(f"ACHP leads (score > 0): {(county_df['Competitive_Score'] > 0).sum():,} "
      f"({(county_df['Competitive_Score'] > 0).mean()*100:.1f}%)")

# ============================================================
# STEP 5 — Corrected HMO-only SNP-excluded Clustering (2026)
# ============================================================
pt_26  = "Plan Type" if "Plan Type" in df26.columns else None
cat_26 = "Contract Category Type" if "Contract Category Type" in df26.columns else None
if "ContractPlanID" not in df26.columns:
    df26["ContractPlanID"] = (df26["Contract ID"].astype(str) + "_" +
                               df26["Plan ID"].astype(str))

if pt_26:
    snp_flag = df26[pt_26].astype(str).str.contains("D-SNP|C-SNP|I-SNP", na=False)
    hmo_flag = df26[pt_26].isin(["HMO", "HMO-POS"])
    snp_cat  = (df26[cat_26] == "SNP") if cat_26 else pd.Series(False, index=df26.index)
    hmo_cl   = df26[hmo_flag & ~snp_flag & ~snp_cat].copy()
else:
    hmo_cl = df26.copy()

if "Organization Marketing Name" in hmo_cl.columns:
    hmo_cl["KP_Flag"] = (hmo_cl["Organization Marketing Name"]
                         .fillna("").str.upper().str.contains("KAISER").astype(int))
else:
    hmo_cl["KP_Flag"] = 0

if "ACHP?" not in hmo_cl.columns:
    achp_col_cl = "ACHP?_2026" if "ACHP?_2026" in hmo_cl.columns else "ACHP?_2025"
    hmo_cl["ACHP?"] = (hmo_cl[achp_col_cl] if achp_col_cl in hmo_cl.columns
                       else pd.Series(0, index=hmo_cl.index))

# Force star rating to numeric in hmo_cl
if "Overall Star Rating" in hmo_cl.columns:
    hmo_cl["Overall Star Rating"] = pd.to_numeric(hmo_cl["Overall Star Rating"], errors="coerce")

print(f"\nNon-SNP HMO/HMO-POS plans: {len(hmo_cl):,}")

CLUSTER_FEATS = (["Monthly Consolidated Premium (Part C + D)",
                   "In-Network Maximum Out-of-Pocket (MOOP) Amount",
                   "Overall Star Rating", "Enrollment"] +
                  [c for c in hmo_cl.columns if c.startswith("Benefit_")])
feats_cl = [f for f in CLUSTER_FEATS if f in hmo_cl.columns]
for f in feats_cl:
    hmo_cl[f] = pd.to_numeric(hmo_cl[f], errors="coerce")

pipe_cl = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
X_cl    = pipe_cl.fit_transform(hmo_cl[feats_cl])

sil_scores = []
for k in range(2, 7):
    lbl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_cl)
    sil_scores.append(silhouette_score(X_cl, lbl, sample_size=min(5000, len(X_cl)), random_state=42))
    print(f"  k={k}  sil={sil_scores[-1]:.4f}")

best_k = 2 + int(np.argmax(sil_scores))
use_k  = max(best_k, 5)
print(f"  Silhouette best k={best_k} → using k={use_k}")

km_cl  = KMeans(n_clusters=use_k, random_state=42, n_init=10)
hmo_cl["Cluster"] = km_cl.fit_predict(X_cl)
pca_cl = PCA(n_components=2, random_state=42)
coords_cl         = pca_cl.fit_transform(X_cl)
ev_cl             = pca_cl.explained_variance_ratio_
hmo_cl["PC1"], hmo_cl["PC2"] = coords_cl[:,0], coords_cl[:,1]
hmo_cl["Cluster_Cohort"] = "National"
hmo_cl.loc[(hmo_cl["ACHP?"] == 1) & (hmo_cl["KP_Flag"] == 0), "Cluster_Cohort"] = "ACHP-nonKP"
hmo_cl.loc[(hmo_cl["ACHP?"] == 1) & (hmo_cl["KP_Flag"] == 1), "Cluster_Cohort"] = "ACHP-KP"

def _cl_profile_row(sub):
    w = sub["Enrollment"].fillna(0)
    return pd.Series({
        "N_Plans":        len(sub),
        "Wtd_Premium":    round(wm(sub["Monthly Consolidated Premium (Part C + D)"], w), 2),
        "Wtd_MOOP":       round(wm(sub["In-Network Maximum Out-of-Pocket (MOOP) Amount"], w), 0),
        "Wtd_Stars":      round(wm(sub["Overall Star Rating"], w), 3),
        "Avg_Enrollment": round(sub["Enrollment"].mean(), 0),
        "ACHP_Pct":       round((sub["ACHP?"] == 1).mean() * 100, 1),
        "KP_Pct":         round((sub["KP_Flag"] == 1).mean() * 100, 1),
    })

cl_sum = hmo_cl.groupby("Cluster").apply(_cl_profile_row).reset_index()
cl_sum["Value_Score"] = (cl_sum["Wtd_Stars"].rank() +
                          cl_sum["Wtd_Premium"].rank(ascending=False) +
                          cl_sum["Wtd_MOOP"].rank(ascending=False))
best_vc = int(cl_sum.loc[cl_sum["Value_Score"].idxmax(), "Cluster"])

cluster_membership_df = (
    hmo_cl.groupby(["Cluster","Cluster_Cohort"])
    .agg(
        Plan_Count=("ContractPlanID", "count"),
        Total_Enrollment=("Enrollment", "sum"),
        Wtd_Stars=("Overall Star Rating",
                   lambda s: wm(s, hmo_cl.loc[s.index, "Enrollment"].fillna(0))),
    )
    .reset_index()
)
cluster_membership_df["Share_of_Cluster_Pct"] = (
    cluster_membership_df["Plan_Count"] /
    cluster_membership_df.groupby("Cluster")["Plan_Count"].transform("sum") * 100
).round(1)
cluster_feature_df = pd.DataFrame({"Feature": feats_cl, "Used_In_HMO_Clustering": 1})

print(cl_sum[["Cluster","N_Plans","Wtd_Stars","Wtd_Premium","Wtd_MOOP","ACHP_Pct","KP_Pct"]].to_string(index=False))
print(f"Best-value cluster: {best_vc}")

# ============================================================
# STEP 6 — Corrected Plan-Level Churn LR
# ============================================================
if "ContractPlanID" not in df25.columns:
    df25["ContractPlanID"] = df25["Contract ID"].astype(str) + "_" + df25["Plan ID"].astype(str)
if "ContractPlanID" not in df26.columns:
    df26["ContractPlanID"] = df26["Contract ID"].astype(str) + "_" + df26["Plan ID"].astype(str)

df25_plan = (df25.sort_values("Enrollment", ascending=False)
               .drop_duplicates(subset="ContractPlanID").reset_index(drop=True))
df26_plan = (df26.sort_values("Enrollment", ascending=False)
               .drop_duplicates(subset="ContractPlanID").reset_index(drop=True))
ids_26 = set(df26_plan["ContractPlanID"].dropna())

df25_plan["Churned"]    = (~df25_plan["ContractPlanID"].isin(ids_26)).astype(int)
df25_plan["KP_Flag_lr"] = (df25_plan.get("Organization Marketing Name", pd.Series("", index=df25_plan.index))
                            .fillna("").str.upper().str.contains("KAISER").astype(int))
df25_plan["Cohort_Label"] = "National"
df25_plan.loc[(df25_plan["ACHP?"] == 1) & (df25_plan["KP_Flag_lr"] == 0), "Cohort_Label"] = "ACHP-nonKP"
df25_plan.loc[(df25_plan["ACHP?"] == 1) & (df25_plan["KP_Flag_lr"] == 1), "Cohort_Label"] = "ACHP-KP"

PT_MAP = {"HMO":0,"HMO-POS":1,"PPO":2,"Regional PPO":3,"HMO D-SNP":4,
          "HMO-POS D-SNP":5,"PPO D-SNP":6,"HMO C-SNP":7,"HMO I-SNP":8,"PFFS":9}
df25_plan["PT_Enc"] = (df25_plan["Plan Type"].map(PT_MAP).fillna(99)
                       if "Plan Type" in df25_plan.columns
                       else pd.Series(99, index=df25_plan.index))

print(f"\n2025 unique plans: {len(df25_plan):,}")
print(f"Exited by 2026:    {df25_plan['Churned'].sum():,} ({df25_plan['Churned'].mean()*100:.1f}%)")

churn_ct = (df25_plan.groupby(["Cohort_Label", "Plan Type"])
            .agg(Plans=("Churned","count"), Exited=("Churned","sum"), Rate=("Churned","mean"))
            .reset_index())
churn_ct["Rate_Pct"] = (churn_ct["Rate"]*100).round(1)

LR_FEATS = (["Monthly Consolidated Premium (Part C + D)",
              "In-Network Maximum Out-of-Pocket (MOOP) Amount",
              "Overall Star Rating", "Enrollment", "ACHP?", "KP_Flag_lr"] +
             [c for c in df25_plan.columns if c.startswith("Benefit_")] + ["PT_Enc"])
feats_lr = [f for f in LR_FEATS if f in df25_plan.columns]
df_lr    = df25_plan[feats_lr + ["Churned"]].copy()
for c in feats_lr:
    df_lr[c] = pd.to_numeric(df_lr[c], errors="coerce")

pipe_lr = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("lr",    LogisticRegression(max_iter=1000, random_state=42,
                                  class_weight="balanced", C=0.5)),
])
auc_cv = cross_val_score(pipe_lr, df_lr[feats_lr].values, df_lr["Churned"].values,
                          scoring="roc_auc",
                          cv=StratifiedKFold(5, shuffle=True, random_state=42))
pipe_lr.fit(df_lr[feats_lr].values, df_lr["Churned"].values)
lr_m    = pipe_lr.named_steps["lr"]
coef_df = (pd.DataFrame({"Feature": feats_lr,
                          "Coefficient": lr_m.coef_[0],
                          "Odds_Ratio":  np.exp(lr_m.coef_[0])})
           .sort_values("Coefficient", ascending=False).reset_index(drop=True))
coef_df["Feature_Label"] = coef_df["Feature"].map(lambda x: FEATURE_LABELS.get(x, x))
print(f"CV AUC: {auc_cv.mean():.4f} ± {auc_cv.std():.4f}")

# ============================================================
# VIZ 1 — 3×3 Structural Vulnerability Heatmap
# ============================================================
pivot = (matrix_3x3_df
         .pivot(index="Plan_Type", columns="Cohort", values="Combined_Cost_Increase")
         .reindex(index=HEATMAP_TYPES, columns=HEATMAP_COHORTS))
worst_row = worst["Plan_Type"] if worst is not None else None
worst_col = worst["Cohort"]    if worst is not None else None

fig_hm, ax_hm = plt.subplots(figsize=(9.8, 5.8))
sns.heatmap(np.zeros_like(pivot, dtype=float), ax=ax_hm,
            cmap=ListedColormap([GREY_BG]), cbar=False,
            linewidths=1.4, linecolor="white")

for i, pt in enumerate(pivot.index):
    for j, cohort in enumerate(pivot.columns):
        val = pivot.loc[pt, cohort]
        label = "n/a" if pd.isna(val) else f"${val:,.0f}"
        is_worst = (pt == worst_row and cohort == worst_col)
        if is_worst:
            ax_hm.add_patch(mpatches.Rectangle((j, i), 1, 1, facecolor=GOLD,
                                                edgecolor="white", linewidth=1.4))
            ax_hm.text(j+0.5, i+0.5, label, ha="center", va="center",
                       fontsize=13, fontweight="bold", color="white")
        else:
            ax_hm.text(j+0.5, i+0.5, label, ha="center", va="center",
                       fontsize=13, fontweight="bold", color=GREY_DARK)

ax_hm.set_xlabel(""); ax_hm.set_ylabel("")
ax_hm.set_yticklabels([PLAN_LABELS.get(p, p) for p in pivot.index], rotation=0, fontsize=FONT_K+1)
ax_hm.set_xticklabels([COHORT_LABELS.get(c, c) for c in pivot.columns], rotation=0, fontsize=FONT_K+1)
ax_hm.tick_params(axis="both", length=0)
for sp in ax_hm.spines.values(): sp.set_visible(False)
if worst_row and worst_col:
    ax_hm.text(pivot.columns.get_loc(worst_col)+0.5,
               pivot.index.get_loc(worst_row)-0.18,
               "Worst cost shift", ha="center", va="bottom",
               fontsize=11, color=GOLD, fontweight="bold")
fig_hm.text(0.5, 0.97, "National HMO-POS plans had the biggest member cost increase",
            ha="center", fontsize=FONT_T+1, fontweight="bold")
fig_hm.text(0.5, 0.93,
            "Each cell shows the enrollment-weighted 2025→2026 increase in premium + MOOP. "
            "Gold cell = clearest competitive opening for ACHP.",
            ha="center", fontsize=11, color=GREY_DARK)
fig_hm.tight_layout(rect=[0,0,1,0.82], pad=2.0)
fig_hm.savefig(f"{OUT3}f3_heatmap.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig_hm)
print("f3_heatmap.png saved")

# ============================================================
# VIZ 2 — County Choropleth
# ============================================================
FIPS_CACHE = HERE / "county_fips_reference.csv"
GJ_CACHE   = HERE / "us_counties.geojson"
FIPS_URL   = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
GJ_URL     = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

try:
    if FIPS_CACHE.exists():
        fips_df = pd.read_csv(FIPS_CACHE, dtype=str)
        if not {"St","StFP","CtyFP","County","Cls"}.issubset(fips_df.columns):
            fips_df = pd.read_csv(FIPS_CACHE, header=None,
                                   names=["St","StFP","CtyFP","County","Cls"], dtype=str)
    else:
        with urllib.request.urlopen(FIPS_URL, timeout=20) as r:
            fips_df = pd.DataFrame(
                [l.split(",") for l in r.read().decode("latin1").strip().splitlines()],
                columns=["St","StFP","CtyFP","County","Cls"])
        fips_df.to_csv(FIPS_CACHE, index=False)
    fips_df["FIPS"] = fips_df["StFP"].str.strip() + fips_df["CtyFP"].str.strip()
    fips_df["County_clean"] = (fips_df["County"].str.strip().str.lower()
        .str.replace(r"\s*(county|parish|borough|census area|municipality|city and borough|municipio)\s*$",
                     "", regex=True).str.strip())
    fips_df["State_clean"] = fips_df["St"].str.strip().str.upper()
    fips_ok = True
except Exception as _fe:
    fips_ok = False; print(f"FIPS fetch failed: {_fe}")

county_df["County_clean"] = (county_df["County Name"].astype(str).str.strip().str.lower()
    .str.replace(r"\s*(county|parish|borough|census area|municipality|city and borough|municipio)\s*$",
                 "", regex=True).str.strip())
county_df["State_clean"] = county_df["State Abbreviation"].astype(str).str.strip().str.upper()

achp_lead_count = 0; total_scored_counties = 0

if fips_ok:
    county_fips = county_df.merge(fips_df[["State_clean","County_clean","FIPS"]],
                                   on=["State_clean","County_clean"], how="left")
    print(f"FIPS matched: {county_fips['FIPS'].notna().sum()}/{len(county_fips)}")
    try:
        if GJ_CACHE.exists():
            with open(GJ_CACHE, "r", encoding="utf-8") as _gf: geo = json.load(_gf)
        else:
            with urllib.request.urlopen(GJ_URL, timeout=30) as _r: geo = json.load(_r)
            with open(GJ_CACHE, "w", encoding="utf-8") as _gf: json.dump(geo, _gf)

        matched               = county_fips.dropna(subset=["FIPS"]).copy()
        achp_lead_count       = int((matched["Competitive_Score"] > 0).sum())
        total_scored_counties = int(len(matched))
        lead_rate             = achp_lead_count / total_scored_counties if total_scored_counties else 0

        feat_lu = {f.get("id"): f for f in geo.get("features", [])}

        def _in_contiguous(feat):
            sfp = str(feat.get("properties",{}).get("STATE","")).zfill(2)
            return sfp not in {"02","15","72","78","60","66","69"}

        def _get_patches(feat):
            geom  = feat.get("geometry",{})
            polys = ([geom["coordinates"]] if geom.get("type")=="Polygon"
                     else geom.get("coordinates",[]) if geom.get("type")=="MultiPolygon" else [])
            out = []
            for poly in polys:
                if not poly: continue
                outer = np.asarray(poly[0], dtype=float)
                if outer.ndim==2 and outer.shape[1]==2:
                    out.append(mpatches.Polygon(outer, closed=True))
            return out

        cont_feats                     = [f for f in geo.get("features",[]) if _in_contiguous(f)]
        bg_patches, lead_patches, lag_patches = [], [], []
        for feat in cont_feats: bg_patches.extend(_get_patches(feat))
        for _, row in matched.iterrows():
            feat = feat_lu.get(row["FIPS"])
            if not feat or not _in_contiguous(feat): continue
            (lead_patches if row["Competitive_Score"] > 0 else lag_patches).extend(_get_patches(feat))

        fig_ch, ax_ch = plt.subplots(figsize=(16, 9))
        ax_ch.add_collection(PatchCollection(bg_patches,   facecolor="#efefef", edgecolor="#d2d2d2", linewidths=0.20, zorder=1))
        ax_ch.add_collection(PatchCollection(lag_patches,  facecolor="#bfc4c9", edgecolor="#7b7f84", linewidths=0.24, zorder=2))
        ax_ch.add_collection(PatchCollection(lead_patches, facecolor=GOLD,      edgecolor="#8B6914", linewidths=0.34, zorder=3))
        ax_ch.autoscale_view(); ax_ch.set_aspect("equal", adjustable="box")
        ax_ch.set_xlim(-125.0, -66.0); ax_ch.set_ylim(24.0, 50.0); ax_ch.set_axis_off()
        ax_ch.set_facecolor("white")
        ax_ch.legend(handles=[
            mpatches.Patch(facecolor=GOLD,     edgecolor="#8B6914", label="ACHP leads in county"),
            mpatches.Patch(facecolor="#bfc4c9", edgecolor="#7b7f84", label="National competitor leads"),
        ], loc="lower left", frameon=False, fontsize=11)
        fig_ch.suptitle(
            f"ACHP leads in {achp_lead_count:,} of {total_scored_counties:,} scored counties ({lead_rate:.0%})",
            fontsize=FONT_T, fontweight="bold", y=0.97)
        ax_ch.text(0.01, 1.01,
                   "Counties where ACHP has the stronger premium / MOOP / stars value are highlighted in gold.",
                   transform=ax_ch.transAxes, fontsize=11, color=GREY_DARK)
        fig_ch.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.03)
        fig_ch.savefig(f"{OUT3}f3_choropleth.png", dpi=220, bbox_inches="tight", facecolor="white")
        plt.show(); plt.close(fig_ch)
        print("f3_choropleth.png saved")
    except Exception as _ge:
        print(f"Choropleth failed: {_ge}")

# ============================================================
# VIZ 3 — PCA Cluster Scatter + Profile Bars
# ============================================================
GREY_SHADES = ["#A8A8A8","#B8B8B8","#C0C0C0","#C8C8C8","#D0D0D0",
               "#D8D8D8","#BEBEBE","#969696","#888888","#909090"]

fig_cl, axes_cl = plt.subplots(1, 2, figsize=(18, 7))
ax_pca = axes_cl[0]
for cl in sorted(hmo_cl["Cluster"].unique()):
    sub    = hmo_cl[hmo_cl["Cluster"]==cl]
    non_kp = sub[sub["KP_Flag"]==0]
    ax_pca.scatter(non_kp["PC1"], non_kp["PC2"],
                   c=GREY_SHADES[cl % len(GREY_SHADES)], s=18, alpha=0.5, label=f"Cluster {cl}")
    kp = sub[sub["KP_Flag"]==1]
    if len(kp) > 0:
        ax_pca.scatter(kp["PC1"], kp["PC2"], c=GOLD, s=80, alpha=0.95, marker="*", zorder=6)
ax_pca.scatter([], [], c=GOLD, s=80, marker="*", label="Kaiser Permanente")
ax_pca.set_xlabel(f"PC1 ({ev_cl[0]:.1%})", fontsize=9)
ax_pca.set_ylabel(f"PC2 ({ev_cl[1]:.1%})", fontsize=9)
ax_pca.set_title(f"HMO/HMO-POS Clusters (k={use_k}) | ★ = Kaiser | SNPs excluded",
                 fontsize=10, fontweight="bold")
ax_pca.legend(fontsize=7, loc="lower right")

ax_prof  = axes_cl[1]; ax_prof2 = ax_prof.twinx()
x_cl = np.arange(use_k); w_cl = 0.22
ax_prof.bar(x_cl-w_cl, cl_sum["Wtd_Stars"],       w_cl, color="#B0B0B0", alpha=0.9,  label="Avg Stars")
ax_prof.bar(x_cl,      cl_sum["Wtd_Premium"]/100,  w_cl, color="#C8C8C8", alpha=0.85, label="Avg Premium (÷100)")
ax_prof.bar(x_cl+w_cl, cl_sum["Wtd_MOOP"]/1000,   w_cl, color="#D8D8D8", alpha=0.85, label="Avg MOOP (÷1000)")
ax_prof2.plot(x_cl, cl_sum["ACHP_Pct"], "o--", color=GOLD,      linewidth=2.5, markersize=9, label="ACHP %")
ax_prof2.plot(x_cl, cl_sum["KP_Pct"],   "s--", color=GREY_DARK, linewidth=1.5, markersize=7, label="KP %")
ax_prof2.set_ylabel("% of Cluster", fontsize=9)
ax_prof2.set_ylim(0, max(cl_sum["ACHP_Pct"].max(), cl_sum["KP_Pct"].max()) * 2.2)
ax_prof.set_xticks(x_cl)
ax_prof.set_xticklabels([f"Cluster {i}" for i in range(use_k)], fontsize=8)
ax_prof.set_title("Cluster Profiles: Stars / Premium / MOOP\n(gold line = ACHP concentration)",
                  fontsize=10, fontweight="bold")
_h1, _l1 = ax_prof.get_legend_handles_labels()
_h2, _l2 = ax_prof2.get_legend_handles_labels()
ax_prof.legend(_h1+_h2, _l1+_l2, fontsize=7.5, loc="upper right")

fig_cl.suptitle(f"Finding 3 — HMO/HMO-POS Cluster Analysis (2026) | k={use_k} | KP isolated",
                fontsize=11, fontweight="bold", y=1.01)
fig_cl.tight_layout()
fig_cl.savefig(f"{OUT3}f3_cluster_viz.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig_cl)
print("f3_cluster_viz.png saved")

# ============================================================
# VIZ 4 — Churn LR: Coefficients + Scenario Bars
# ============================================================
main_types_ch = ["HMO","HMO-POS","PPO","Regional PPO"]
churn_plot    = (churn_ct[churn_ct["Plan Type"].isin(main_types_ch)].copy()
                 if "Plan Type" in churn_ct.columns else pd.DataFrame())
pt_col_25     = "Plan Type" if "Plan Type" in churn_plot.columns else None

ncols   = 3 if (len(churn_plot) > 0 and pt_col_25) else 2
fig_lr, axes_lr = plt.subplots(1, ncols, figsize=(18, 6), gridspec_kw={"wspace": 0.42})

ax_a       = axes_lr[0]
focal_feats = {"ACHP?"}
colors_lr   = [GOLD if f in focal_feats else GREY_DARK for f in coef_df["Feature"]]
ax_a.barh(coef_df["Feature_Label"], coef_df["Coefficient"], color=colors_lr, alpha=0.9, edgecolor="white")
ax_a.axvline(0, color="#555", linewidth=0.9, linestyle="--")
ax_a.set_xlabel("Log-Odds Coefficient", fontsize=9)
ax_a.set_title("Churn Risk Drivers\n(Gold = ACHP effect | Grey = controls)", fontsize=10, fontweight="bold")
ax_a.tick_params(labelsize=7.5); ax_a.invert_yaxis()

if len(churn_plot) > 0 and pt_col_25:
    ax_b     = axes_lr[1]
    pivot_ch = (churn_plot.pivot(index=pt_col_25, columns="Cohort_Label", values="Rate_Pct")
                .fillna(0).reindex(columns=["ACHP-KP","ACHP-nonKP","National"], fill_value=0))
    sns.heatmap(pivot_ch, ax=ax_b, annot=True, fmt=".1f", cmap="RdYlGn_r",
                linewidths=0.5, annot_kws={"size":9,"weight":"bold"},
                cbar_kws={"label":"Exit Rate (%)","shrink":0.7})
    ax_b.set_title("Plan Exit Rate 2025→2026\nby Plan Type × Cohort (%)", fontsize=10, fontweight="bold")
    ax_b.set_yticklabels(ax_b.get_yticklabels(), rotation=0, fontsize=8.5)
    last_idx = 2
else:
    last_idx = 1

ax_c       = axes_lr[last_idx]
_feat_base = {f: 0 for f in feats_lr}
if len(feats_lr) >= 4:
    _scen_list = [
        {**_feat_base, feats_lr[0]: 0,   feats_lr[1]: 3400, feats_lr[2]: 4.5,
         feats_lr[3]: 5000,  "ACHP?": 1, "KP_Flag_lr": 0, "PT_Enc": 0, "_label": "ACHP HMO\n★4.5 $0"},
        {**_feat_base, feats_lr[0]: 10,  feats_lr[1]: 4000, feats_lr[2]: 4.5,
         feats_lr[3]: 50000, "ACHP?": 1, "KP_Flag_lr": 1, "PT_Enc": 0, "_label": "ACHP-KP HMO\n★4.5"},
        {**_feat_base, feats_lr[0]: 30,  feats_lr[1]: 5500, feats_lr[2]: 3.5,
         feats_lr[3]: 1500,  "ACHP?": 0, "KP_Flag_lr": 0, "PT_Enc": 0, "_label": "Natl HMO\n★3.5 $30"},
        {**_feat_base, feats_lr[0]: 100, feats_lr[1]: 7500, feats_lr[2]: 3.0,
         feats_lr[3]: 500,   "ACHP?": 0, "KP_Flag_lr": 0, "PT_Enc": 2, "_label": "Natl PPO\n★3.0 $100"},
    ]
    _scen_df     = pd.DataFrame(_scen_list)
    _scen_labels = _scen_df.pop("_label").tolist()
    _scen_X      = _scen_df[[f for f in feats_lr if f in _scen_df.columns]].fillna(0).values
    _probs       = pipe_lr.predict_proba(_scen_X)[:, 1]
    _bar_colors  = [GOLD if "ACHP" in lbl else GREY_DARK for lbl in _scen_labels]
    _bars = ax_c.bar(range(len(_scen_list)), _probs*100, color=_bar_colors, alpha=0.9, width=0.6)
    for _b, _v in zip(_bars, _probs*100):
        ax_c.text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.4,
                  f"{_v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax_c.set_xticks(range(len(_scen_list)))
    ax_c.set_xticklabels(_scen_labels, fontsize=7.5)
    ax_c.set_ylabel("Predicted Exit Probability (%)", fontsize=9)
    ax_c.set_title(f"Predicted Churn — Plan Scenarios\nCV AUC={auc_cv.mean():.3f}", fontsize=10, fontweight="bold")
    ax_c.set_ylim(0, max(_probs*100)*1.25)
    ax_c.legend(handles=[mpatches.Patch(color=GOLD,      label="ACHP"),
                          mpatches.Patch(color=GREY_DARK, label="National")], fontsize=8, frameon=False)
else:
    ax_c.text(0.5, 0.5, "Need 4+ LR features", ha="center", va="center",
              transform=ax_c.transAxes); ax_c.axis("off")

fig_lr.suptitle(f"Finding 3 — Plan Churn Analysis (2025→2026) | CV AUC={auc_cv.mean():.3f}",
                fontsize=11, fontweight="bold", y=1.02)
fig_lr.tight_layout()
fig_lr.savefig(f"{OUT3}f3_churn_lr.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig_lr)
print("f3_churn_lr.png saved")

# ============================================================
# STEP 7 — Save to Excel (12 sheets)
# ============================================================
xl_out = f"{OUT3}finding3_results_v2.xlsx"

def _map_labels(df, col, mapping):
    df = df.copy(); df[col] = df[col].map(lambda x: mapping.get(x, x)); return df

with pd.ExcelWriter(xl_out, engine="openpyxl") as _w:
    _map_labels(crosstab_df, "Plan_Type", PLAN_LABELS).pipe(
        lambda d: _map_labels(d, "Cohort", COHORT_LABELS)).to_excel(_w, sheet_name="plan_type_cohort_crosstab", index=False)
    _map_labels(matrix_3x3_df, "Plan_Type", PLAN_LABELS).pipe(
        lambda d: _map_labels(d, "Cohort", COHORT_LABELS)).to_excel(_w, sheet_name="plan_type_cohort_3x3", index=False)
    county_df.drop(columns=["County_clean","State_clean"], errors="ignore").to_excel(
        _w, sheet_name="county_competitive_analysis", index=False)
    _map_labels(kaiser_df, "Group", GROUP_LABELS).to_excel(_w, sheet_name="kaiser_isolation", index=False)
    hmo_cl[["ContractPlanID","Plan Type","ACHP?","KP_Flag","Cluster",
             "Monthly Consolidated Premium (Part C + D)",
             "In-Network Maximum Out-of-Pocket (MOOP) Amount",
             "Overall Star Rating","Enrollment","PC1","PC2"]].to_excel(
        _w, sheet_name="hmo_cluster_assignments", index=False)
    cl_sum.to_excel(_w, sheet_name="hmo_cluster_summary", index=False)
    _map_labels(cluster_membership_df, "Cluster_Cohort", COHORT_LABELS).to_excel(
        _w, sheet_name="hmo_cluster_membership", index=False)
    cluster_feature_df.to_excel(_w, sheet_name="hmo_cluster_features", index=False)
    coef_df.to_excel(_w, sheet_name="churn_lr_coefs", index=False)
    _map_labels(churn_ct, "Cohort_Label", COHORT_LABELS).to_excel(_w, sheet_name="churn_by_cohort_plantype", index=False)
    yoy_sum = []
    for _coh in ["ACHP-KP","ACHP-nonKP","Regional","National"]:
        _s  = yoy[yoy["Cohort"]==_coh]; _ws = _s[W].fillna(0)
        yoy_sum.append({
            "Cohort":               COHORT_LABELS.get(_coh, _coh),
            "N_Rows":               len(_s),
            "Total_Enrollment":     int(_ws.sum()),
            "Wtd_Stars_2026":       round(wm(_s[star26], _ws), 3) if star26 in _s.columns else np.nan,
            "Wtd_Delta_Premium":    round(wm(_s["Delta_Premium"],    _ws), 2),
            "Wtd_Delta_MOOP":       round(wm(_s["Delta_MOOP"],       _ws), 2),
            "Wtd_Delta_Deductible": round(wm(_s["Delta_Deductible"], _ws), 2),
            "Pct_Plans_Raised_Premium": round((_s["Delta_Premium"].fillna(0)>0).mean()*100, 1),
            "Pct_Plans_Raised_MOOP":    round((_s["Delta_MOOP"].fillna(0)>0).mean()*100, 1),
        })
    pd.DataFrame(yoy_sum).to_excel(_w, sheet_name="yoy_cohort_summary", index=False)
    county_df.sort_values("Competitive_Score", ascending=False).to_excel(
        _w, sheet_name="county_competitive_ranked", index=False)

print(f"\nfinding3_results_v2.xlsx saved (12 sheets)")
print("\n" + "="*55)
print("Finding 3 complete.")
print(f"  f3_heatmap.png          — 3×3 structural vulnerability")
print(f"  f3_choropleth.png       — county competitive map")
print(f"  f3_cluster_viz.png      — HMO cluster PCA + profiles")
print(f"  f3_churn_lr.png         — churn drivers + scenarios")
print(f"  finding3_results_v2.xlsx — 12-sheet workbook")

Cohort distribution:
Cohort
National      95600
Regional       6850
ACHP-nonKP     6160
ACHP-KP         405

Worst cell: HMO-POS × National = $475
   Plan_Type     Cohort  Plan_Count  Wtd_Delta_Premium  Wtd_Delta_MOOP  Combined_Cost_Increase
         HMO    ACHP-KP         184              11.13          215.61                  226.74
         HMO ACHP-nonKP        1165               1.76          269.88                  271.65
         HMO   Regional           0                NaN             NaN                     NaN
         HMO   National       10218               0.42           35.20                   35.62
     HMO-POS    ACHP-KP         140               5.89          295.21                  301.10
     HMO-POS ACHP-nonKP         954               8.43          256.48                  264.91
     HMO-POS   Regional           0                NaN             NaN                     NaN
     HMO-POS   National       15333               2.52          472.47                  474.9

## Cell 19 — Output Summary

In [22]:
print('='*65)
print('OUTPUT FILES')
print('='*65)
outputs = [
    ('finding1_viz_ml_output.xlsx',           'F1: Charts + regression tables'),
    ('chart1_grouped_bar.png',                'F1: Weighted delta premium & MOOP'),
    ('chart2_diverging_dist.png',             'F1: Plan-level premium distributions'),
    ('chart3_scatter.png',                    'F1: 2025 vs 2026 premium scatter'),
    ('chart4_regression_coefs.png',           'F1: OLS coefficient plot'),
    ('finding2_benefit_erosion.png',          'F2: % plans with drug benefit erosion'),
    ('finding2_deductible_increase.png',      'F2: Weighted delta deductible'),
    ('finding2_scatter.png',                  'F2: Top 10 nationals scatter'),
    ('finding3_elbow.png',                    'F3: Cluster validation metrics'),
    ('finding3_viz_snp_excluded_clusters.png','F3: All clusters + KP overlay'),
    ('finding3_viz_churn_lr.png',             'F3: Churn logistic regression'),
]
for fname, desc in outputs:
    status = 'OK' if os.path.exists(f'/content/{fname}') else 'MISSING'
    print(f'  [{status}]  {fname:<45}  {desc}')

print()
print('='*65)
print('PROFESSOR CRITIQUE FIXES APPLIED')
print('='*65)
print('''
1. DEDUPLICATION BUG
   Fixed: 4-column merge key preserves all plan-county rows.
   No drop_duplicates anywhere in pipeline.

2. ENROLLMENT-WEIGHTED STAR RATINGS
   Fixed: All cohort star summaries use weighted_mean()
   with Enrollment_2025 as weights. Labeled in chart titles.

3. KAISER PERMANENTE ISOLATION
   Fixed: KP_Flag detects Kaiser in org/contract name.
   Cohort = ACHP-KP / ACHP-nonKP / National throughout.
   Finding 3 cluster chart shows KP% per cluster as overlay.

4. ALL CLUSTERS SHOWN (Finding 3)
   Fixed: All k clusters shown in PCA scatter and bar chart.
   Feature list documented in cluster summary printout.

5. YoY FRAMING
   All three findings are 2025->2026 delta analysis.
   Ensure Slide 6 of presentation leads with this framing.
''')


OUTPUT FILES
  [OK]  finding1_viz_ml_output.xlsx                    F1: Charts + regression tables
  [OK]  chart1_grouped_bar.png                         F1: Weighted delta premium & MOOP
  [OK]  chart2_diverging_dist.png                      F1: Plan-level premium distributions
  [OK]  chart3_scatter.png                             F1: 2025 vs 2026 premium scatter
  [OK]  chart4_regression_coefs.png                    F1: OLS coefficient plot
  [OK]  finding2_benefit_erosion.png                   F2: % plans with drug benefit erosion
  [OK]  finding2_deductible_increase.png               F2: Weighted delta deductible
  [OK]  finding2_scatter.png                           F2: Top 10 nationals scatter
  [OK]  finding3_elbow.png                             F3: Cluster validation metrics
  [OK]  finding3_viz_snp_excluded_clusters.png         F3: All clusters + KP overlay
  [MISSING]  finding3_viz_churn_lr.png                      F3: Churn logistic regression

PROFESSOR CRITIQUE FIXES APP